# FREIGHT-MNet Strategic Cold-OD Baselines

This notebook is a **standalone baseline runner** for the FREIGHT-MNet / D-CQHGT project.  It is designed to add the strategic baselines that are most useful for a KDD/ICDM-style main paper and appendix, while keeping the output schema directly comparable with the existing Cold-OD, GraphSAGE/HGT, topology, disruption, calibration, and CVaR notebooks.

The notebook follows the current project data contract:

```text
E:\NetworkOptimization\pythonProject1\Data
├── 08_processed\model_ready
│   ├── freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet
│   └── _metadata\freight_mnet_supervised_feature_manifest.json
├── 08_processed\graph_inputs
│   ├── freight_mnet_full_heterodata_east_plus_gulf.pt
│   ├── topology_features_od_east_plus_gulf.parquet
│   └── freight_mnet_full_heterodata_east_plus_gulf.metadata.json
└── 10_experiments
    ├── cold_od_split_baselines_v1_notebook\east_plus_gulf\predictions_cold_od_val_test.parquet
    ├── graphsage_hgt_cold_od_baselines_v2_notebook\east_plus_gulf\combined_predictions_graph_cold_od_val_test_v2.parquet
    ├── dcqhgt_topology_cold_od_v3_ablation_notebook\east_plus_gulf\combined_predictions_dcqhgt_topology_v3_ablation_val_test.parquet
    └── dcqhgt_disruption_gate_stabilization_v1_notebook\east_plus_gulf\combined_predictions_disruption_gate_stabilized.parquet
```

## Baselines implemented

Main-paper strategic baselines:

1. **ID-MLP / STID-style quantile baseline** — origin embedding, destination embedding, year embedding, current OD features, and a residual monotone quantile head. This directly tests whether the graph gain is actually a zone-identity / edge-decoder effect.
2. **ZoneID-only MLP** — origin/destination/year embeddings only, with no numeric OD features. This is a stronger identity-control baseline than a pure global prior.
3. **OD-GCN-lite quantile baseline** — a simple FAF-zone homogeneous GCN encoder plus OD edge decoder. This is the adapted OD-matrix-style graph baseline.
4. **R-GCN quantile baseline** — a relation-aware graph baseline over the typed freight graph, less expressive than HGT but more relation-aware than homogeneous GraphSAGE/GCN.
5. **CQR-LGBM quantile baseline** — LightGBM quantile regression with validation residual calibration. This is the strong tabular probabilistic baseline.
6. **Decision-ranking baselines** — Cold-prior ranking, LGBM-q75 ranking, demand-volume ranking, and random top-k ranking for CVaR-style screening.

Appendix baselines are optional and controlled by `cfg.enabled_appendix_baselines`.

## Reproducibility and restart design

Long-running parts are **checkpointed** under `10_experiments/strategic_cold_od_baselines_v1_notebook/<scope>/models/`. Neural baselines save `last.pt` after every epoch and best checkpoints for validation pinball, q75 MAE, and IQR MAE. LightGBM baselines save per-quantile model pickle files. If `cfg.resume=True`, completed or partially completed runs are reused.

Progress from training is printed using **single-line console refresh** so the notebook output stays compact even when running many seeds.


In [1]:
import sys
import torch

print("Python executable:", sys.executable)
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

Python executable: D:\environment\python\python12\python.exe
Torch version: 2.5.1+cu121
CUDA available: True
CUDA version: 12.1
GPU count: 1
GPU name: NVIDIA GeForce RTX 4050 Laptop GPU


## 1. Environment setup and optional dependency preflight

The notebook keeps hard dependencies small. PyTorch is required for neural baselines. PyTorch Geometric is required for OD-GCN-lite and R-GCN. LightGBM is required for CQR-LGBM. Missing optional packages skip only the baselines that depend on them; the notebook still writes a manifest explaining what was skipped.


In [1]:
from __future__ import annotations

import gc
import json
import math
import os
import pickle
import random
import shutil
import sys
import time
import warnings
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

# Keep notebook output stable on mixed Windows/Anaconda environments.
os.environ.setdefault("NUMEXPR_MAX_THREADS", "8")
os.environ.setdefault("PANDAS_USE_NUMEXPR", "0")
os.environ.setdefault("PYTHONHASHSEED", "42")

import numpy as np
import pandas as pd

pd.set_option("compute.use_numexpr", False)
pd.set_option("compute.use_bottleneck", False)
warnings.filterwarnings("default")

try:
    import torch
    from torch import nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except Exception as exc:  # pragma: no cover - dependency guard
    torch = None
    nn = None
    F = None
    DataLoader = None
    TensorDataset = None
    TORCH_AVAILABLE = False
    print(f"[optional] PyTorch is unavailable; neural baselines will be skipped. Error: {exc}")

try:
    from torch_geometric.data import HeteroData
    from torch_geometric.nn import GCNConv, RGCNConv
    PYG_AVAILABLE = True
except Exception as exc:  # pragma: no cover - dependency guard
    HeteroData = None
    GCNConv = None
    RGCNConv = None
    PYG_AVAILABLE = False
    print(f"[optional] PyTorch Geometric is unavailable; graph baselines will be skipped. Error: {exc}")

try:
    import lightgbm as lgb
    LGB_AVAILABLE = True
except Exception as exc:  # pragma: no cover - dependency guard
    lgb = None
    LGB_AVAILABLE = False
    print(f"[optional] LightGBM is unavailable; CQR-LGBM baselines will be skipped. Error: {exc}")

print("Python:", sys.version)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("torch available:", TORCH_AVAILABLE)
if TORCH_AVAILABLE:
    print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
print("PyG available:", PYG_AVAILABLE)
print("LightGBM available:", LGB_AVAILABLE)


[optional] PyTorch Geometric is unavailable; graph baselines will be skipped. Error: No module named 'torch_geometric'
[optional] LightGBM is unavailable; CQR-LGBM baselines will be skipped. Error: No module named 'lightgbm'
Python: 3.12.8 (tags/v3.12.8:2dc476b, Dec  3 2024, 19:30:04) [MSC v.1942 64 bit (AMD64)]
pandas: 2.2.3
numpy: 2.2.3
torch available: True
torch: 2.5.1+cu121 cuda: True
PyG available: False
LightGBM available: False


## 2. Configuration

Only this cell should normally need editing. The default path uses the current project structure, but it can be overridden by setting the environment variable `FREIGHT_MNET_DATA_ROOT` before running the notebook.

For a smoke test, set `cfg.seeds=(7,)`, `cfg.max_epochs=5`, and reduce `cfg.enabled_main_baselines`.


In [2]:
@dataclass
class ExperimentConfig:
    """Central configuration for strategic Cold-OD baselines."""

    # Project paths and scope.
    data_root: Path = Path(os.environ.get("FREIGHT_MNET_DATA_ROOT", r"E:\NetworkOptimization\pythonProject1\Data"))
    scope: str = "east_plus_gulf"
    run_name: str = "strategic_cold_od_baselines_v1_notebook"

    # Cold-OD split fallback settings. Existing stored masks are preferred.
    train_years: Tuple[int, ...] = (2018, 2019, 2020, 2021, 2022)
    val_year: int = 2023
    test_year: int = 2024
    split_seed: int = 42
    val_pair_fraction: float = 0.10
    test_pair_fraction: float = 0.10

    # Main and appendix baseline selection.
    enabled_main_baselines: Tuple[str, ...] = (
        "cold_prior",
        "id_mlp_stid",
        "zone_id_only_mlp",
        "od_gcn_lite",
        "rgcn",
        "cqr_lgbm",
    )
    enabled_appendix_baselines: Tuple[str, ...] = (
        "numeric_mlp_no_id",
        "topology_only_lgbm",
        "demand_only_lgbm",
    )
    run_appendix_baselines: bool = True

    # Neural training setup.
    seeds: Tuple[int, ...] = (7, 42, 123, 2026, 535)
    max_epochs: int = 300
    patience: int = 50
    batch_size: int = 4096
    hidden_dim: int = 128
    embedding_dim: int = 64
    graph_layers: int = 2
    dropout: float = 0.10
    learning_rate: float = 1.0e-3
    weight_decay: float = 1.0e-4
    grad_clip_norm: float = 5.0
    lambda_iqr: float = 0.10

    # LightGBM setup.
    lgbm_n_estimators: int = 5000
    lgbm_learning_rate: float = 0.03
    lgbm_num_leaves: int = 63
    lgbm_min_child_samples: int = 30
    lgbm_subsample: float = 0.90
    lgbm_colsample_bytree: float = 0.90
    lgbm_early_stopping_rounds: int = 150

    # Weighting and metrics.
    label_columns: Tuple[str, ...] = ("truck_q25", "truck_q50", "truck_q75")
    weight_column: str = "obs_weight_sum"
    weight_clip_min: float = 0.05
    weight_clip_max: float = 20.0
    cvar_eta: float = 0.50
    cvar_top_fractions: Tuple[float, ...] = (0.05, 0.10, 0.20)
    random_rank_repeats: int = 100

    # Execution behavior.
    resume: bool = True
    overwrite: bool = False
    save_models: bool = True
    make_plots: bool = False
    progress_min_interval_sec: float = 1.0
    device: str = "cuda" if (TORCH_AVAILABLE and torch.cuda.is_available()) else "cpu"


cfg = ExperimentConfig()
print(cfg)


ExperimentConfig(data_root=WindowsPath('E:/NetworkOptimization/pythonProject1/Data'), scope='east_plus_gulf', run_name='strategic_cold_od_baselines_v1_notebook', train_years=(2018, 2019, 2020, 2021, 2022), val_year=2023, test_year=2024, split_seed=42, val_pair_fraction=0.1, test_pair_fraction=0.1, enabled_main_baselines=('cold_prior', 'id_mlp_stid', 'zone_id_only_mlp', 'od_gcn_lite', 'rgcn', 'cqr_lgbm'), enabled_appendix_baselines=('numeric_mlp_no_id', 'topology_only_lgbm', 'demand_only_lgbm'), run_appendix_baselines=True, seeds=(7, 42, 123, 2026, 535), max_epochs=300, patience=50, batch_size=4096, hidden_dim=128, embedding_dim=64, graph_layers=2, dropout=0.1, learning_rate=0.001, weight_decay=0.0001, grad_clip_norm=5.0, lambda_iqr=0.1, lgbm_n_estimators=5000, lgbm_learning_rate=0.03, lgbm_num_leaves=63, lgbm_min_child_samples=30, lgbm_subsample=0.9, lgbm_colsample_bytree=0.9, lgbm_early_stopping_rounds=150, label_columns=('truck_q25', 'truck_q50', 'truck_q75'), weight_column='obs_weig

## 3. Path management and artifact layout

The output layout mirrors prior notebooks: predictions, metrics, leaderboards, CVaR tables, histories, models, and run metadata are all written under one experiment directory.


In [3]:
@dataclass(frozen=True)
class ExperimentPaths:
    """Resolved paths used by this notebook."""

    data_root: Path
    model_ready_dir: Path
    graph_inputs_dir: Path
    experiment_root: Path
    supervised_path: Path
    manifest_path: Path
    heterodata_path: Path
    topology_features_path: Path
    metadata_path: Path
    output_dir: Path
    models_dir: Path
    tables_dir: Path
    plots_dir: Path
    reports_dir: Path
    external_prediction_paths: Mapping[str, Path]


def build_paths(config: ExperimentConfig) -> ExperimentPaths:
    """Create all project paths from the central configuration."""
    data_root = Path(config.data_root)
    model_ready_dir = data_root / "08_processed" / "model_ready"
    graph_inputs_dir = data_root / "08_processed" / "graph_inputs"
    experiment_root = data_root / "10_experiments"
    output_dir = experiment_root / config.run_name / config.scope

    external_prediction_paths = {
        "ColdOD_NonGraph": experiment_root / "cold_od_split_baselines_v1_notebook" / config.scope / "predictions_cold_od_val_test.parquet",
        "GraphV2_SAGE_HGT": experiment_root / "graphsage_hgt_cold_od_baselines_v2_notebook" / config.scope / "combined_predictions_graph_cold_od_val_test_v2.parquet",
        "DCQHGT_TopologyV3": experiment_root / "dcqhgt_topology_cold_od_v3_ablation_notebook" / config.scope / "combined_predictions_dcqhgt_topology_v3_ablation_val_test.parquet",
        "DCQHGT_DisruptionGate": experiment_root / "dcqhgt_disruption_gate_stabilization_v1_notebook" / config.scope / "combined_predictions_disruption_gate_stabilized.parquet",
    }

    return ExperimentPaths(
        data_root=data_root,
        model_ready_dir=model_ready_dir,
        graph_inputs_dir=graph_inputs_dir,
        experiment_root=experiment_root,
        supervised_path=model_ready_dir / f"freight_mnet_supervised_edges_2018_2024_{config.scope}.parquet",
        manifest_path=model_ready_dir / "_metadata" / "freight_mnet_supervised_feature_manifest.json",
        heterodata_path=graph_inputs_dir / f"freight_mnet_full_heterodata_{config.scope}.pt",
        topology_features_path=graph_inputs_dir / f"topology_features_od_{config.scope}.parquet",
        metadata_path=graph_inputs_dir / f"freight_mnet_full_heterodata_{config.scope}.metadata.json",
        output_dir=output_dir,
        models_dir=output_dir / "models",
        tables_dir=output_dir / "tables",
        plots_dir=output_dir / "plots",
        reports_dir=output_dir / "reports",
        external_prediction_paths=external_prediction_paths,
    )


paths = build_paths(cfg)
for directory in [paths.output_dir, paths.models_dir, paths.tables_dir, paths.plots_dir, paths.reports_dir]:
    directory.mkdir(parents=True, exist_ok=True)

print("Output directory:", paths.output_dir)
print("Supervised path:", paths.supervised_path)
print("HeteroData path:", paths.heterodata_path)


Output directory: E:\NetworkOptimization\pythonProject1\Data\10_experiments\strategic_cold_od_baselines_v1_notebook\east_plus_gulf
Supervised path: E:\NetworkOptimization\pythonProject1\Data\08_processed\model_ready\freight_mnet_supervised_edges_2018_2024_east_plus_gulf.parquet
HeteroData path: E:\NetworkOptimization\pythonProject1\Data\08_processed\graph_inputs\freight_mnet_full_heterodata_east_plus_gulf.pt


## 4. Utility functions

These helpers are intentionally small and reusable. They cover seeding, atomic writes, single-line progress, schema normalization, Parquet safety, and feature preprocessing.


In [4]:
LABEL_COLUMNS = list(cfg.label_columns)
TAUS = np.array([0.25, 0.50, 0.75], dtype=np.float32)
SUPERVISED_EDGE_TYPE = ("faf_zone", "supervised_od", "faf_zone")
TOPOLOGY_FEATURE_PREFIX = "topo_"
FMI_DIAGNOSTIC_COLUMNS = {
    "n_fmi_county_pairs",
    "obs_weight_sum",
    "input_q25_min",
    "input_q25_max",
    "input_q50_min",
    "input_q50_max",
    "input_q75_min",
    "input_q75_max",
}
COLD_PRIOR_COLUMNS = [
    "cold_prior_truck_q25",
    "cold_prior_truck_q50",
    "cold_prior_truck_q75",
    "cold_prior_iqr",
    "cold_prior_q75_q50_gap",
    "cold_prior_q50_q25_gap",
    "cold_has_origin_prior",
    "cold_has_dest_prior",
    "cold_has_any_zone_prior",
]


def set_global_seed(seed: int) -> None:
    """Set Python, NumPy, and PyTorch random seeds."""
    random.seed(seed)
    np.random.seed(seed)
    if TORCH_AVAILABLE:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True


class ProgressPrinter:
    """Single-line progress printer for long notebook cells."""

    def __init__(self, min_interval_sec: float = 1.0):
        self.min_interval_sec = float(min_interval_sec)
        self._last_time = 0.0
        self._last_len = 0

    def update(self, message: str, force: bool = False) -> None:
        now = time.time()
        if (not force) and (now - self._last_time < self.min_interval_sec):
            return
        self._last_time = now
        clean = str(message).replace("\n", " ")
        pad = " " * max(0, self._last_len - len(clean))
        print("\r" + clean + pad, end="", flush=True)
        self._last_len = len(clean)

    def done(self, message: str = "done") -> None:
        self.update(message, force=True)
        print(flush=True)
        self._last_len = 0


def normalize_faf_code(value: Any) -> str:
    """Normalize a FAF zone code as a three-character string when possible."""
    if pd.isna(value):
        return ""
    text = str(value).strip()
    if text.endswith(".0"):
        text = text[:-2]
    if text.isdigit():
        return text.zfill(3)
    return text


def infer_column(df: pd.DataFrame, candidates: Sequence[str], purpose: str) -> str:
    """Return the first available column from a list of candidates."""
    for column in candidates:
        if column in df.columns:
            return column
    raise KeyError(f"Could not infer {purpose}. Tried: {list(candidates)}")


def ensure_file(path: Path, description: str, required: bool = True) -> bool:
    """Check whether an input file exists and optionally raise a clear error."""
    exists = Path(path).exists()
    if not exists and required:
        raise FileNotFoundError(f"Missing {description}: {path}")
    if not exists:
        print(f"[optional] Missing {description}: {path}")
    return exists


def atomic_write_json(payload: Mapping[str, Any], path: Path) -> None:
    """Write JSON atomically to avoid partial metadata files after interruption."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2, default=str)
    tmp.replace(path)


def atomic_pickle_dump(obj: Any, path: Path) -> None:
    """Write a pickle file atomically."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("wb") as file:
        pickle.dump(obj, file)
    tmp.replace(path)


def safe_to_parquet(frame: pd.DataFrame, path: Path) -> None:
    """Write a DataFrame to Parquet after normalizing object/list columns."""
    path.parent.mkdir(parents=True, exist_ok=True)
    clean = frame.copy()
    for col in clean.columns:
        if clean[col].dtype == "object":
            sample = clean[col].dropna().head(20).tolist()
            if any(isinstance(x, (dict, list, tuple, set)) for x in sample):
                clean[col] = clean[col].map(lambda x: json.dumps(x, default=str) if isinstance(x, (dict, list, tuple, set)) else x)
    clean.to_parquet(path, index=False, engine="pyarrow")


def write_dataframe(frame: pd.DataFrame, path: Path) -> None:
    """Write CSV or Parquet based on suffix."""
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".parquet":
        safe_to_parquet(frame, path)
    else:
        frame.to_csv(path, index=False)


def storage_has_key(storage: Any, key: str) -> bool:
    """Return True if a PyG storage object contains a key."""
    try:
        return key in storage.keys()
    except Exception:
        return hasattr(storage, key)


def as_numpy_bool(x: Any, expected_len: Optional[int] = None) -> np.ndarray:
    """Convert tensors/arrays/lists to a boolean NumPy array."""
    if TORCH_AVAILABLE and isinstance(x, torch.Tensor):
        arr = x.detach().cpu().numpy()
    else:
        arr = np.asarray(x)
    arr = arr.astype(bool)
    if expected_len is not None and len(arr) != expected_len:
        raise ValueError(f"Boolean mask length mismatch: got {len(arr)}, expected {expected_len}")
    return arr


class NumericPreprocessor:
    """Median-impute and z-score numeric feature columns using train rows only."""

    def __init__(self, columns: Sequence[str]):
        self.columns = list(columns)
        self.median_: Optional[np.ndarray] = None
        self.mean_: Optional[np.ndarray] = None
        self.std_: Optional[np.ndarray] = None

    def fit(self, frame: pd.DataFrame, mask: np.ndarray) -> "NumericPreprocessor":
        if len(self.columns) == 0:
            self.median_ = np.zeros(0, dtype=np.float32)
            self.mean_ = np.zeros(0, dtype=np.float32)
            self.std_ = np.ones(0, dtype=np.float32)
            return self
        values = frame.loc[mask, self.columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
        med = np.nanmedian(values, axis=0)
        med = np.where(np.isfinite(med), med, 0.0)
        values = np.where(np.isfinite(values), values, med)
        mean = values.mean(axis=0)
        std = values.std(axis=0)
        std = np.where(std > 1.0e-6, std, 1.0)
        self.median_ = med.astype(np.float32)
        self.mean_ = mean.astype(np.float32)
        self.std_ = std.astype(np.float32)
        return self

    def transform(self, frame: pd.DataFrame) -> np.ndarray:
        if self.median_ is None or self.mean_ is None or self.std_ is None:
            raise RuntimeError("NumericPreprocessor must be fitted before transform().")
        if len(self.columns) == 0:
            return np.zeros((len(frame), 0), dtype=np.float32)
        values = frame[self.columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
        values = np.where(np.isfinite(values), values, self.median_)
        out = (values - self.mean_) / self.std_
        out[~np.isfinite(out)] = 0.0
        return out.astype(np.float32)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "columns": self.columns,
            "median": None if self.median_ is None else self.median_.tolist(),
            "mean": None if self.mean_ is None else self.mean_.tolist(),
            "std": None if self.std_ is None else self.std_.tolist(),
        }


## 5. Load supervised OD-year table, manifest, optional topology features, and optional HeteroData

The supervised table is the source of labels and row-level metadata. The feature manifest is the source of leakage-safe current OD/zone features. The HeteroData artifact is required only for graph baselines.


In [5]:
ensure_file(paths.supervised_path, "supervised model-ready parquet", required=True)
ensure_file(paths.manifest_path, "feature manifest JSON", required=True)

with paths.manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)
manifest_feature_columns = list(manifest.get("feature_columns", []))
if not manifest_feature_columns:
    raise ValueError("Feature manifest does not contain a non-empty 'feature_columns' list.")

supervised_df = pd.read_parquet(paths.supervised_path).reset_index(drop=True)
if "row_id" not in supervised_df.columns:
    supervised_df.insert(0, "row_id", np.arange(len(supervised_df), dtype=np.int64))
else:
    supervised_df["row_id"] = pd.to_numeric(supervised_df["row_id"], errors="raise").astype(np.int64)

origin_col = infer_column(supervised_df, ["faf_orig", "faf_orig_str", "origin", "orig_faf", "o_faf"], "origin FAF column")
dest_col = infer_column(supervised_df, ["faf_dest", "faf_dest_str", "destination", "dest_faf", "d_faf"], "destination FAF column")
year_col = infer_column(supervised_df, ["year", "year_int", "target_year"], "year column")

supervised_df["faf_orig_str"] = supervised_df[origin_col].map(normalize_faf_code)
supervised_df["faf_dest_str"] = supervised_df[dest_col].map(normalize_faf_code)
supervised_df["year_int"] = pd.to_numeric(supervised_df[year_col], errors="raise").astype(int)

missing_labels = [c for c in LABEL_COLUMNS if c not in supervised_df.columns]
if missing_labels:
    raise ValueError(f"Missing label columns: {missing_labels}")
for col in LABEL_COLUMNS:
    supervised_df[col] = pd.to_numeric(supervised_df[col], errors="coerce").astype(float)

# Optional topology features from the graph-input build notebook.
topology_feature_columns: List[str] = []
if ensure_file(paths.topology_features_path, "topology OD feature parquet", required=False):
    topology_df = pd.read_parquet(paths.topology_features_path).copy()
    if "row_id" not in topology_df.columns:
        raise ValueError("Topology feature table exists but does not contain row_id.")
    topology_feature_columns = [c for c in topology_df.columns if c.startswith(TOPOLOGY_FEATURE_PREFIX)]
    supervised_df = supervised_df.merge(
        topology_df[["row_id"] + topology_feature_columns],
        on="row_id",
        how="left",
        validate="one_to_one",
    )
    print(f"Loaded topology features: {len(topology_feature_columns)} columns")
else:
    print("No topology feature table found; topology-only baselines will be skipped.")

# Optional HeteroData object.
hetero_data = None
if PYG_AVAILABLE and ensure_file(paths.heterodata_path, "full PyG HeteroData artifact", required=False):
    try:
        hetero_data = torch.load(paths.heterodata_path, map_location="cpu", weights_only=False)
    except TypeError:
        hetero_data = torch.load(paths.heterodata_path, map_location="cpu")
    if not isinstance(hetero_data, HeteroData):
        raise TypeError(f"Loaded heterodata path is not HeteroData: {type(hetero_data)}")
    print("Loaded HeteroData:")
    print(hetero_data)
else:
    print("HeteroData is unavailable; graph baselines will be skipped.")

print("Supervised table shape:", supervised_df.shape)
print("Manifest feature count:", len(manifest_feature_columns))
print("Years:", sorted(supervised_df["year_int"].unique().tolist()))
print("First columns:", supervised_df.columns[:20].tolist())


Loaded topology features: 16 columns
HeteroData is unavailable; graph baselines will be skipped.
Supervised table shape: (73972, 112)
Manifest feature count: 64
Years: [2018, 2019, 2020, 2021, 2022, 2023, 2024]
First columns: ['row_id', 'scope', 'faf_orig', 'faf_dest', 'faf_od', 'year', 'split', 'truck_q25', 'truck_q50', 'truck_q75', 'truck_iqr', 'truck_iqr_over_q50', 'n_fmi_county_pairs', 'obs_weight_sum', 'obs_weight_mean', 'obs_weight_max', 'demand_merge_status', 'orig_faf_zone_name', 'dest_faf_zone_name', 'dest_east_plus_gulf_county_share']


## 6. Cold-OD split reconstruction and leakage-safe fallback priors

The preferred split source is the HeteroData supervised edge store, because it was created by the graph-input build notebook and stores `cold_train_mask`, `cold_val_mask`, and `cold_test_mask`. If HeteroData is unavailable, the notebook falls back to an existing Cold-OD prediction file. Only if neither exists does it rebuild a deterministic pair-level split.

The prior used here is **not** an exact OD historical prior for Cold-OD test pairs. It blends train-only origin-zone and destination-zone medians, then falls back to global train medians.


In [6]:
def cold_split_from_heterodata(data: Any, n_rows: int) -> Optional[pd.Series]:
    """Extract row-level Cold-OD split labels from HeteroData masks if available."""
    if data is None or SUPERVISED_EDGE_TYPE not in data.edge_types:
        return None
    store = data[SUPERVISED_EDGE_TYPE]
    required = ["cold_train_mask", "cold_val_mask", "cold_test_mask"]
    if not all(storage_has_key(store, key) for key in required):
        return None

    row_ids = None
    if storage_has_key(store, "row_id"):
        raw = store.row_id
        row_ids = raw.detach().cpu().numpy().astype(int) if TORCH_AVAILABLE and isinstance(raw, torch.Tensor) else np.asarray(raw, dtype=int)
    else:
        row_ids = np.arange(n_rows, dtype=int)
    if len(row_ids) != n_rows:
        print(f"Warning: HeteroData row_id length {len(row_ids)} differs from supervised rows {n_rows}; attempting row_id alignment.")

    split = pd.Series(["unused"] * n_rows, index=np.arange(n_rows), dtype="object")
    for name in ["train", "val", "test", "unused"]:
        key = f"cold_{name}_mask"
        if storage_has_key(store, key):
            mask = as_numpy_bool(getattr(store, key))
            ids = row_ids[mask]
            ids = ids[(ids >= 0) & (ids < n_rows)]
            split.iloc[ids] = name
    return split


def normalize_prediction_keys(frame: pd.DataFrame) -> pd.DataFrame:
    """Normalize OD-year keys in an external prediction table."""
    out = frame.copy()
    rename = {
        "origin": "faf_orig_str",
        "faf_orig": "faf_orig_str",
        "orig_faf": "faf_orig_str",
        "destination": "faf_dest_str",
        "faf_dest": "faf_dest_str",
        "dest_faf": "faf_dest_str",
        "year": "year_int",
        "target_year": "year_int",
        "cold_od_split": "split",
        "cold_split": "split",
    }
    out = out.rename(columns={k: v for k, v in rename.items() if k in out.columns})
    if "faf_orig_str" in out.columns:
        out["faf_orig_str"] = out["faf_orig_str"].map(normalize_faf_code)
    if "faf_dest_str" in out.columns:
        out["faf_dest_str"] = out["faf_dest_str"].map(normalize_faf_code)
    if "year_int" in out.columns:
        out["year_int"] = pd.to_numeric(out["year_int"], errors="coerce").astype("Int64")
    return out


def cold_split_from_external_predictions(frame: pd.DataFrame, external_path: Path) -> Optional[pd.Series]:
    """Infer Cold-OD split labels from an all-splits or val/test prediction file."""
    if not external_path.exists():
        return None
    pred = normalize_prediction_keys(pd.read_parquet(external_path))
    if "split" not in pred.columns:
        return None
    needed = ["faf_orig_str", "faf_dest_str", "year_int", "split"]
    if not all(c in pred.columns for c in needed):
        return None
    keys = pred[needed].drop_duplicates()
    dup = keys.groupby(["faf_orig_str", "faf_dest_str", "year_int"])["split"].nunique()
    if int((dup > 1).sum()) > 0:
        raise ValueError(f"External split file has conflicting split labels: {external_path}")
    merged = frame[["faf_orig_str", "faf_dest_str", "year_int"]].merge(
        keys,
        on=["faf_orig_str", "faf_dest_str", "year_int"],
        how="left",
    )
    return merged["split"].fillna("unused").astype(str).str.lower()


def deterministic_cold_split(frame: pd.DataFrame, config: ExperimentConfig) -> pd.Series:
    """Build a deterministic pair-level Cold-OD split when no stored split exists."""
    rng = np.random.default_rng(config.split_seed)
    pair_frame = frame[["faf_orig_str", "faf_dest_str"]].drop_duplicates().reset_index(drop=True)
    pair_keys = pair_frame["faf_orig_str"].astype(str) + "->" + pair_frame["faf_dest_str"].astype(str)
    pairs = pair_keys.to_numpy()
    rng.shuffle(pairs)

    n_test = max(1, int(round(len(pairs) * config.test_pair_fraction)))
    n_val = max(1, int(round(len(pairs) * config.val_pair_fraction)))
    test_pairs = set(pairs[:n_test])
    val_pairs = set(pairs[n_test:n_test + n_val])

    row_pair = frame["faf_orig_str"].astype(str) + "->" + frame["faf_dest_str"].astype(str)
    year = frame["year_int"].to_numpy()
    split = pd.Series(["unused"] * len(frame), dtype="object")
    train_mask = np.isin(year, np.array(config.train_years, dtype=int)) & (~row_pair.isin(val_pairs | test_pairs).to_numpy())
    val_mask = (year == int(config.val_year)) & row_pair.isin(val_pairs).to_numpy()
    test_mask = (year == int(config.test_year)) & row_pair.isin(test_pairs).to_numpy()
    split.loc[train_mask] = "train"
    split.loc[val_mask] = "val"
    split.loc[test_mask] = "test"
    return split


# Choose the best available Cold-OD split source.
split_series = cold_split_from_heterodata(hetero_data, len(supervised_df))
split_source = "heterodata_masks"

if split_series is None:
    # Prefer all-splits file when available; otherwise val/test files only recover val/test.
    candidate_paths = [
        paths.experiment_root / "cold_od_split_baselines_v1_notebook" / cfg.scope / "predictions_cold_od_all_splits.parquet",
        paths.external_prediction_paths["ColdOD_NonGraph"],
    ]
    for candidate in candidate_paths:
        split_series = cold_split_from_external_predictions(supervised_df, candidate)
        if split_series is not None:
            split_source = f"external_predictions:{candidate}"
            break

if split_series is None:
    split_series = deterministic_cold_split(supervised_df, cfg)
    split_source = "deterministic_rebuild"

supervised_df["cold_split"] = split_series.astype(str).str.lower().replace({"nan": "unused"})
mask_dict = {name: supervised_df["cold_split"].eq(name).to_numpy() for name in ["train", "val", "test"]}

for split_name, mask in mask_dict.items():
    if int(mask.sum()) == 0:
        raise ValueError(f"Cold-OD split '{split_name}' is empty. Split source: {split_source}")

print("Cold split source:", split_source)
print(supervised_df["cold_split"].value_counts(dropna=False).to_string())

# Write split statistics for paper table / appendix.
split_stats = []
for split_name in ["train", "val", "test", "unused"]:
    part = supervised_df.loc[supervised_df["cold_split"].eq(split_name)]
    split_stats.append({
        "split": split_name,
        "n_rows": int(len(part)),
        "n_od_pairs": int((part["faf_orig_str"] + "->" + part["faf_dest_str"]).nunique()) if len(part) else 0,
        "years": ",".join(map(str, sorted(part["year_int"].unique().tolist()))) if len(part) else "",
        "n_origin_zones": int(part["faf_orig_str"].nunique()) if len(part) else 0,
        "n_dest_zones": int(part["faf_dest_str"].nunique()) if len(part) else 0,
    })
split_stats_df = pd.DataFrame(split_stats)
split_stats_df.to_csv(paths.tables_dir / "strategic_baselines_cold_od_split_stats.csv", index=False)
split_stats_df


Cold split source: external_predictions:E:\NetworkOptimization\pythonProject1\Data\10_experiments\cold_od_split_baselines_v1_notebook\east_plus_gulf\predictions_cold_od_all_splits.parquet
cold_split
train     42849
unused    29109
test       1057
val         957


,split,n_rows,n_od_pairs,years,n_origin_zones,n_dest_zones
0,train,42849,8748,"2018,2019,2020,2021,2022",104,104
1,val,957,957,2023,104,104
2,test,1057,1057,2024,104,104
3,unused,29109,10631,"2018,2019,2020,2021,2022,2023,2024",104,104


In [7]:
def compute_zone_prior_maps(frame: pd.DataFrame, train_mask: np.ndarray) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray], np.ndarray]:
    """Compute train-only origin, destination, and global median quantile priors."""
    train = frame.loc[train_mask].copy()
    global_prior = train[LABEL_COLUMNS].median().to_numpy(dtype=np.float32)
    origin_map = {
        normalize_faf_code(k): v[LABEL_COLUMNS].median().to_numpy(dtype=np.float32)
        for k, v in train.groupby("faf_orig_str")
    }
    dest_map = {
        normalize_faf_code(k): v[LABEL_COLUMNS].median().to_numpy(dtype=np.float32)
        for k, v in train.groupby("faf_dest_str")
    }
    return origin_map, dest_map, global_prior


def apply_zone_priors(
    rows: pd.DataFrame,
    origin_map: Mapping[str, np.ndarray],
    dest_map: Mapping[str, np.ndarray],
    global_prior: np.ndarray,
) -> pd.DataFrame:
    """Apply an origin/destination fallback prior to a dataframe slice."""
    out = []
    for row in rows.itertuples(index=False):
        origin = normalize_faf_code(getattr(row, "faf_orig_str"))
        dest = normalize_faf_code(getattr(row, "faf_dest_str"))
        origin_vec = origin_map.get(origin)
        dest_vec = dest_map.get(dest)
        has_o = origin_vec is not None
        has_d = dest_vec is not None
        if has_o and has_d:
            pred = 0.5 * (origin_vec + dest_vec)
        elif has_o:
            pred = origin_vec
        elif has_d:
            pred = dest_vec
        else:
            pred = global_prior
        pred = np.asarray(pred, dtype=np.float32)
        out.append({
            "cold_prior_truck_q25": float(pred[0]),
            "cold_prior_truck_q50": float(pred[1]),
            "cold_prior_truck_q75": float(pred[2]),
            "cold_prior_iqr": float(pred[2] - pred[0]),
            "cold_prior_q75_q50_gap": float(pred[2] - pred[1]),
            "cold_prior_q50_q25_gap": float(pred[1] - pred[0]),
            "cold_has_origin_prior": float(has_o),
            "cold_has_dest_prior": float(has_d),
            "cold_has_any_zone_prior": float(has_o or has_d),
        })
    return pd.DataFrame(out, index=rows.index)


def build_oof_cold_priors(frame: pd.DataFrame, n_folds: int = 5, seed: int = 42) -> pd.DataFrame:
    """Build leakage-safe train OOF priors and full-train priors for validation/test rows."""
    result = pd.DataFrame(index=frame.index, columns=COLD_PRIOR_COLUMNS, dtype=float)
    train_idx = np.where(frame["cold_split"].eq("train").to_numpy())[0]
    rng = np.random.default_rng(seed)
    shuffled = train_idx.copy()
    rng.shuffle(shuffled)
    folds = np.array_split(shuffled, max(2, min(n_folds, len(shuffled))))

    for fold_id, fold_idx in enumerate(folds):
        if len(fold_idx) == 0:
            continue
        fit_mask = frame["cold_split"].eq("train").to_numpy()
        fit_mask[fold_idx] = False
        origin_map, dest_map, global_prior = compute_zone_prior_maps(frame, fit_mask)
        result.loc[fold_idx, COLD_PRIOR_COLUMNS] = apply_zone_priors(frame.loc[fold_idx], origin_map, dest_map, global_prior).to_numpy()

    full_origin_map, full_dest_map, full_global_prior = compute_zone_prior_maps(frame, frame["cold_split"].eq("train").to_numpy())
    non_train_idx = np.where(~frame["cold_split"].eq("train").to_numpy())[0]
    if len(non_train_idx):
        result.loc[non_train_idx, COLD_PRIOR_COLUMNS] = apply_zone_priors(
            frame.loc[non_train_idx], full_origin_map, full_dest_map, full_global_prior
        ).to_numpy()

    if result.isna().any().any():
        raise ValueError("Cold fallback prior construction produced NaN values.")
    return result.astype(np.float32)


prior_features = build_oof_cold_priors(supervised_df, n_folds=5, seed=cfg.split_seed)
for col in COLD_PRIOR_COLUMNS:
    supervised_df[col] = prior_features[col].astype(np.float32)

print(supervised_df.loc[supervised_df["cold_split"].isin(["train", "val", "test"]), COLD_PRIOR_COLUMNS].describe().T)


                           count         mean         std          min  \
cold_prior_truck_q25     44863.0  1556.373901  197.760162  1114.102539   
cold_prior_truck_q50     44863.0  2313.678467  303.581085  1558.235107   
cold_prior_truck_q75     44863.0  3712.536377  459.233459  2726.409912   
cold_prior_iqr           44863.0  2156.162598  315.996185  1503.212402   
cold_prior_q75_q50_gap   44863.0  1398.858154  208.930099   751.270020   
cold_prior_q50_q25_gap   44863.0   757.304688  137.738708   364.007446   
cold_has_origin_prior    44863.0     1.000000    0.000000     1.000000   
cold_has_dest_prior      44863.0     1.000000    0.000000     1.000000   
cold_has_any_zone_prior  44863.0     1.000000    0.000000     1.000000   

                                 25%          50%          75%          max  
cold_prior_truck_q25     1403.022522  1533.445068  1684.325012  2395.217529  
cold_prior_truck_q50     2097.263672  2309.612549  2530.792480  3223.142578  
cold_prior_truck_q75     

## 7. Feature matrices, labels, weights, and graph tensors

This cell builds the common numeric matrices used by all baselines. The same target scaling, sample weights, and split indices are used across neural models. Graph-specific tensors are created only when HeteroData is available.


In [8]:
def make_sample_weights(frame: pd.DataFrame, config: ExperimentConfig) -> np.ndarray:
    """Build train-normalized log support weights for weighted pinball loss."""
    if config.weight_column in frame.columns:
        raw = pd.to_numeric(frame[config.weight_column], errors="coerce").fillna(0.0).to_numpy(dtype=np.float32)
        weights = np.log1p(np.maximum(raw, 0.0))
    else:
        weights = np.ones(len(frame), dtype=np.float32)
    weights = np.clip(weights, config.weight_clip_min, config.weight_clip_max)
    train_mean = float(np.mean(weights[mask_dict["train"]])) if mask_dict["train"].any() else float(np.mean(weights))
    if not np.isfinite(train_mean) or train_mean <= 0:
        train_mean = 1.0
    return (weights / train_mean).astype(np.float32)


feature_columns_base = [c for c in manifest_feature_columns if c in supervised_df.columns]
missing_manifest_cols = sorted(set(manifest_feature_columns) - set(feature_columns_base))
if missing_manifest_cols:
    print("Warning: some manifest columns are missing and will be ignored:", missing_manifest_cols[:20])

# Exclude labels and diagnostics from predictive feature blocks.
feature_columns_base = [c for c in feature_columns_base if c not in LABEL_COLUMNS and c not in FMI_DIAGNOSTIC_COLUMNS]
base_plus_prior_columns = feature_columns_base + COLD_PRIOR_COLUMNS
base_plus_topology_columns = base_plus_prior_columns + topology_feature_columns

# Appendix feature groups.
def select_keyword_columns(columns: Sequence[str], keywords: Sequence[str]) -> List[str]:
    selected = []
    for col in columns:
        low = col.lower()
        if any(keyword in low for keyword in keywords):
            selected.append(col)
    return selected

demand_feature_columns = select_keyword_columns(
    feature_columns_base,
    ["tons", "ton", "value", "truck", "rail", "water", "air", "pipeline", "demand", "flow", "faf5", "mode", "commodity"],
)
if not demand_feature_columns:
    print("Warning: no demand-like feature columns were found by keyword; demand-only LGBM will be skipped.")

preprocessors: Dict[str, NumericPreprocessor] = {
    "base_plus_prior": NumericPreprocessor(base_plus_prior_columns).fit(supervised_df, mask_dict["train"]),
    "base_plus_topology": NumericPreprocessor(base_plus_topology_columns).fit(supervised_df, mask_dict["train"]),
    "topology_plus_prior": NumericPreprocessor(topology_feature_columns + COLD_PRIOR_COLUMNS).fit(supervised_df, mask_dict["train"]) if topology_feature_columns else NumericPreprocessor([]).fit(supervised_df, mask_dict["train"]),
    "demand_plus_prior": NumericPreprocessor(demand_feature_columns + COLD_PRIOR_COLUMNS).fit(supervised_df, mask_dict["train"]) if demand_feature_columns else NumericPreprocessor([]).fit(supervised_df, mask_dict["train"]),
    "empty": NumericPreprocessor([]).fit(supervised_df, mask_dict["train"]),
}

X_base_prior = preprocessors["base_plus_prior"].transform(supervised_df)
X_base_topology = preprocessors["base_plus_topology"].transform(supervised_df)
X_topology_prior = preprocessors["topology_plus_prior"].transform(supervised_df)
X_demand_prior = preprocessors["demand_plus_prior"].transform(supervised_df)
X_empty = preprocessors["empty"].transform(supervised_df)

sample_weight_np = make_sample_weights(supervised_df, cfg)
y_np = supervised_df[LABEL_COLUMNS].to_numpy(dtype=np.float32)
base_prior_np = supervised_df[["cold_prior_truck_q25", "cold_prior_truck_q50", "cold_prior_truck_q75"]].to_numpy(dtype=np.float32)

target_scale = float(np.nanmedian(y_np[mask_dict["train"], 1]))
if not np.isfinite(target_scale) or target_scale <= 1.0:
    target_scale = 1000.0

y_scaled_np = (y_np / target_scale).astype(np.float32)
base_prior_scaled_np = (base_prior_np / target_scale).astype(np.float32)

# Zone and year indices for embedding baselines. HeteroData edge_label_index is preferred for consistency with graph baselines.
zone_values = sorted(set(supervised_df["faf_orig_str"].astype(str)) | set(supervised_df["faf_dest_str"].astype(str)))
zone_to_idx = {zone: idx for idx, zone in enumerate(zone_values)}
origin_idx_np = supervised_df["faf_orig_str"].map(zone_to_idx).to_numpy(dtype=np.int64)
dest_idx_np = supervised_df["faf_dest_str"].map(zone_to_idx).to_numpy(dtype=np.int64)

edge_label_index_np = None
if hetero_data is not None and SUPERVISED_EDGE_TYPE in hetero_data.edge_types:
    supervised_store = hetero_data[SUPERVISED_EDGE_TYPE]
    if storage_has_key(supervised_store, "edge_label_index"):
        raw_edge_label_index = supervised_store.edge_label_index.detach().cpu().numpy().astype(np.int64)
        if raw_edge_label_index.shape[1] == len(supervised_df):
            edge_label_index_np = raw_edge_label_index
            origin_idx_np = edge_label_index_np[0]
            dest_idx_np = edge_label_index_np[1]
            zone_values = [f"faf_node_{i}" for i in range(int(max(origin_idx_np.max(), dest_idx_np.max()) + 1))]
            zone_to_idx = {zone: i for i, zone in enumerate(zone_values)}

unique_years = sorted(supervised_df["year_int"].astype(int).unique().tolist())
year_to_idx = {year: idx for idx, year in enumerate(unique_years)}
year_idx_np = supervised_df["year_int"].map(year_to_idx).to_numpy(dtype=np.int64)

split_indices_np = {name: np.where(mask)[0].astype(np.int64) for name, mask in mask_dict.items()}

print("Feature dimensions:")
print("  base + cold prior:", X_base_prior.shape)
print("  base + cold prior + topology:", X_base_topology.shape)
print("  topology + prior:", X_topology_prior.shape)
print("  demand + prior:", X_demand_prior.shape)
print("  target scale:", target_scale)
print("  zones for embeddings:", int(max(origin_idx_np.max(), dest_idx_np.max()) + 1))
print("  years:", year_to_idx)


Feature dimensions:
  base + cold prior: (73972, 73)
  base + cold prior + topology: (73972, 89)
  topology + prior: (73972, 25)
  demand + prior: (73972, 50)
  target scale: 2317.030029296875
  zones for embeddings: 104
  years: {2018: 0, 2019: 1, 2020: 2, 2021: 3, 2022: 4, 2023: 5, 2024: 6}


In [9]:
def standardize_node_features(x: Any) -> np.ndarray:
    """Return z-scored node features as a NumPy float32 matrix."""
    if TORCH_AVAILABLE and isinstance(x, torch.Tensor):
        arr = x.detach().cpu().numpy().astype(np.float32)
    else:
        arr = np.asarray(x, dtype=np.float32)
    mean = np.nanmean(arr, axis=0)
    std = np.nanstd(arr, axis=0)
    mean = np.where(np.isfinite(mean), mean, 0.0)
    std = np.where(np.isfinite(std) & (std > 1.0e-6), std, 1.0)
    arr = np.where(np.isfinite(arr), arr, mean)
    out = (arr - mean) / std
    out[~np.isfinite(out)] = 0.0
    return out.astype(np.float32)


def make_bidirectional_edge_index(edge_index: np.ndarray) -> np.ndarray:
    """Duplicate same-type edges in reverse direction and remove duplicates."""
    if edge_index.size == 0:
        return edge_index.reshape(2, 0).astype(np.int64)
    rev = edge_index[::-1, :]
    combined = np.concatenate([edge_index, rev], axis=1)
    pairs = pd.DataFrame(combined.T, columns=["src", "dst"]).drop_duplicates()
    return pairs[["src", "dst"]].to_numpy(dtype=np.int64).T


def get_edge_index_np(data: Any, edge_type: Tuple[str, str, str], bidirectional_same_type: bool = False) -> Optional[np.ndarray]:
    """Extract one edge_index array from HeteroData."""
    if data is None or edge_type not in data.edge_types or not storage_has_key(data[edge_type], "edge_index"):
        return None
    edge_index = data[edge_type].edge_index.detach().cpu().numpy().astype(np.int64)
    if bidirectional_same_type and edge_type[0] == edge_type[2]:
        edge_index = make_bidirectional_edge_index(edge_index)
    return edge_index


def build_homogeneous_faf_edge_index(data: Any, include_topology: bool = False) -> Optional[np.ndarray]:
    """Build a homogeneous FAF-zone graph for OD-GCN-lite."""
    if data is None:
        return None
    arrays = []
    relations = [
        ("faf_zone", "train_od", "faf_zone"),
        ("faf_zone", "rev_train_od", "faf_zone"),
        ("faf_zone", "demand_truck", "faf_zone"),
        ("faf_zone", "rev_demand_truck", "faf_zone"),
        ("faf_zone", "demand_rail", "faf_zone"),
        ("faf_zone", "rev_demand_rail", "faf_zone"),
        ("faf_zone", "demand_multimodal", "faf_zone"),
        ("faf_zone", "rev_demand_multimodal", "faf_zone"),
        ("faf_zone", "self_loop", "faf_zone"),
    ]
    if include_topology:
        relations.extend([
            ("faf_zone", "truck_adj", "faf_zone"),
            ("faf_zone", "rail_adj", "faf_zone"),
        ])
    for edge_type in relations:
        arr = get_edge_index_np(data, edge_type, bidirectional_same_type=edge_type[1] in {"truck_adj", "rail_adj"})
        if arr is not None and arr.shape[1] > 0:
            arrays.append(arr)
    if not arrays:
        return None
    combined = np.concatenate(arrays, axis=1)
    pairs = pd.DataFrame(combined.T, columns=["src", "dst"]).drop_duplicates()
    return pairs[["src", "dst"]].to_numpy(dtype=np.int64).T


def build_rgcn_graph(data: Any) -> Optional[Dict[str, Any]]:
    """Flatten typed HeteroData into a single R-GCN graph with relation IDs."""
    if data is None:
        return None
    node_offsets: Dict[str, int] = {}
    x_parts: List[np.ndarray] = []
    offset = 0
    for node_type in data.node_types:
        if not storage_has_key(data[node_type], "x"):
            continue
        x = standardize_node_features(data[node_type].x)
        # Projecting to a common dimension happens in the model; here we pad to max dimension.
        node_offsets[node_type] = offset
        x_parts.append(x)
        offset += x.shape[0]
    if not x_parts or "faf_zone" not in node_offsets:
        return None
    max_dim = max(part.shape[1] for part in x_parts)
    padded = []
    for part in x_parts:
        if part.shape[1] < max_dim:
            pad = np.zeros((part.shape[0], max_dim - part.shape[1]), dtype=np.float32)
            part = np.concatenate([part, pad], axis=1)
        padded.append(part)
    x_all = np.concatenate(padded, axis=0).astype(np.float32)

    edge_arrays = []
    edge_types = []
    relation_to_id = {}
    for edge_type in data.edge_types:
        if edge_type == SUPERVISED_EDGE_TYPE:
            continue
        if not storage_has_key(data[edge_type], "edge_index"):
            continue
        src_type, relation, dst_type = edge_type
        if src_type not in node_offsets or dst_type not in node_offsets:
            continue
        edge_index = data[edge_type].edge_index.detach().cpu().numpy().astype(np.int64).copy()
        if edge_index.shape[1] == 0:
            continue
        edge_index[0, :] += node_offsets[src_type]
        edge_index[1, :] += node_offsets[dst_type]
        rel_id = relation_to_id.setdefault(str(edge_type), len(relation_to_id))
        edge_arrays.append(edge_index)
        edge_types.append(np.full(edge_index.shape[1], rel_id, dtype=np.int64))
    if not edge_arrays:
        return None
    edge_index_all = np.concatenate(edge_arrays, axis=1).astype(np.int64)
    edge_type_all = np.concatenate(edge_types).astype(np.int64)
    faf_offset = node_offsets["faf_zone"]
    supervised_edge_index_global = np.vstack([origin_idx_np + faf_offset, dest_idx_np + faf_offset]).astype(np.int64)
    return {
        "x_all": x_all,
        "edge_index": edge_index_all,
        "edge_type": edge_type_all,
        "num_relations": len(relation_to_id),
        "node_offsets": node_offsets,
        "relation_to_id": relation_to_id,
        "supervised_edge_index_global": supervised_edge_index_global,
    }


faf_node_features_np = None
hom_faf_edge_index_np = None
rgcn_graph = None
if hetero_data is not None and PYG_AVAILABLE:
    if "faf_zone" in hetero_data.node_types and storage_has_key(hetero_data["faf_zone"], "x"):
        faf_node_features_np = standardize_node_features(hetero_data["faf_zone"].x)
    hom_faf_edge_index_np = build_homogeneous_faf_edge_index(hetero_data, include_topology=False)
    rgcn_graph = build_rgcn_graph(hetero_data)

print("Graph tensors:")
print("  FAF node features:", None if faf_node_features_np is None else faf_node_features_np.shape)
print("  Homogeneous FAF edge index:", None if hom_faf_edge_index_np is None else hom_faf_edge_index_np.shape)
print("  R-GCN graph:", None if rgcn_graph is None else {k: (v.shape if hasattr(v, 'shape') else v) for k, v in rgcn_graph.items() if k != 'relation_to_id'})


Graph tensors:
  FAF node features: None
  Homogeneous FAF edge index: None
  R-GCN graph: None


## 8. Metrics, calibration, and prediction-frame schema

All baselines are converted to the same prediction table schema, so they can be concatenated with previous experiment outputs without additional cleaning.


In [10]:
def enforce_monotone_numpy(pred: np.ndarray) -> np.ndarray:
    """Clip predictions to non-negative values and enforce q25 <= q50 <= q75."""
    arr = np.asarray(pred, dtype=np.float32)
    arr = np.where(np.isfinite(arr), arr, 0.0)
    arr = np.maximum(arr, 0.0)
    return np.sort(arr, axis=1).astype(np.float32)


def pinball_matrix(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """Return pinball loss with shape [n, 3]."""
    err = y_true - y_pred
    taus = TAUS.reshape(1, 3)
    return np.maximum(taus * err, (taus - 1.0) * err)


def make_prediction_frame(
    row_indices: np.ndarray,
    pred: np.ndarray,
    source: str,
    model: str,
    checkpoint: str,
    seed: Any = "baseline",
) -> pd.DataFrame:
    """Create the common prediction table for selected row indices."""
    idx = np.asarray(row_indices, dtype=np.int64)
    pred = enforce_monotone_numpy(pred)
    base = supervised_df.iloc[idx].copy()
    out = pd.DataFrame({
        "source": source,
        "model": model,
        "checkpoint": checkpoint,
        "seed": str(seed),
        "split": base["cold_split"].to_numpy(),
        "row_id": base["row_id"].astype(int).to_numpy(),
        "faf_orig_str": base["faf_orig_str"].astype(str).to_numpy(),
        "faf_dest_str": base["faf_dest_str"].astype(str).to_numpy(),
        "year_int": base["year_int"].astype(int).to_numpy(),
        "true_q25": base["truck_q25"].astype(float).to_numpy(),
        "true_q50": base["truck_q50"].astype(float).to_numpy(),
        "true_q75": base["truck_q75"].astype(float).to_numpy(),
        "pred_q25": pred[:, 0].astype(float),
        "pred_q50": pred[:, 1].astype(float),
        "pred_q75": pred[:, 2].astype(float),
        "sample_weight": sample_weight_np[idx].astype(float),
    })
    if "n_fmi_county_pairs" in supervised_df.columns:
        out["n_fmi_county_pairs"] = pd.to_numeric(base["n_fmi_county_pairs"], errors="coerce").to_numpy(float)
    else:
        out["n_fmi_county_pairs"] = np.nan
    return out


def compute_metrics_frame(prediction_frame: pd.DataFrame) -> pd.DataFrame:
    """Compute comparable metrics grouped by source/model/checkpoint/seed/split."""
    rows = []
    if prediction_frame.empty:
        return pd.DataFrame()
    group_cols = ["source", "model", "checkpoint", "seed", "split"]
    for keys, group in prediction_frame.groupby(group_cols, dropna=False):
        source, model, checkpoint, seed, split = keys
        y_true = group[["true_q25", "true_q50", "true_q75"]].to_numpy(float)
        y_pred = group[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(float)
        weights = group["sample_weight"].to_numpy(float) if "sample_weight" in group.columns else np.ones(len(group))
        weights = np.where(np.isfinite(weights), weights, 1.0)
        losses = pinball_matrix(y_true, y_pred)
        row_pinball = losses.mean(axis=1)
        abs_err = np.abs(y_pred - y_true)
        true_iqr = y_true[:, 2] - y_true[:, 0]
        pred_iqr = y_pred[:, 2] - y_pred[:, 0]
        iqr_abs = np.abs(pred_iqr - true_iqr)
        q75_threshold = np.quantile(y_true[:, 2], 0.90) if len(group) else np.nan
        iqr_threshold = np.quantile(true_iqr, 0.90) if len(group) else np.nan
        top_q75 = y_true[:, 2] >= q75_threshold if len(group) else np.zeros(0, dtype=bool)
        top_iqr = true_iqr >= iqr_threshold if len(group) else np.zeros(0, dtype=bool)
        support = group["n_fmi_county_pairs"].to_numpy(float) if "n_fmi_county_pairs" in group.columns else np.full(len(group), np.nan)
        if np.isfinite(support).sum() > 0 and len(np.unique(support[np.isfinite(support)])) > 1:
            support_threshold = np.nanquantile(support, 0.25)
            sparse = support <= support_threshold
        else:
            support_threshold = np.nan
            sparse = np.zeros(len(group), dtype=bool)

        def masked_mean(values: np.ndarray, mask: np.ndarray) -> float:
            return float(np.mean(values[mask])) if np.any(mask) else np.nan

        rows.append({
            "source": source,
            "model": model,
            "checkpoint": checkpoint,
            "seed": seed,
            "split": split,
            "n_rows": int(len(group)),
            "pinball_mean": float(np.mean(row_pinball)),
            "weighted_pinball_mean": float(np.average(row_pinball, weights=weights)),
            "mae_q25": float(np.mean(abs_err[:, 0])),
            "mae_q50": float(np.mean(abs_err[:, 1])),
            "mae_q75": float(np.mean(abs_err[:, 2])),
            "iqr_mae": float(np.mean(iqr_abs)),
            "bias_q25": float(np.mean(y_pred[:, 0] - y_true[:, 0])),
            "bias_q50": float(np.mean(y_pred[:, 1] - y_true[:, 1])),
            "bias_q75": float(np.mean(y_pred[:, 2] - y_true[:, 2])),
            "stress_top10_threshold_q75": float(q75_threshold),
            "stress_top10_mae_q75": masked_mean(abs_err[:, 2], top_q75),
            "top_iqr10_threshold": float(iqr_threshold),
            "top_iqr10_mae_q75": masked_mean(abs_err[:, 2], top_iqr),
            "top_iqr10_iqr_mae": masked_mean(iqr_abs, top_iqr),
            "sparse_bottom25_threshold": float(support_threshold) if np.isfinite(support_threshold) else np.nan,
            "sparse_bottom25_mae_q75": masked_mean(abs_err[:, 2], sparse),
            "raw_crossing_rate": float(np.mean((y_pred[:, 0] > y_pred[:, 1]) | (y_pred[:, 1] > y_pred[:, 2]))),
        })
    return pd.DataFrame(rows)


def select_best_calibration(raw_frame: pd.DataFrame, source: str, model: str, seed: Any = "baseline") -> pd.DataFrame:
    """Create validation-selected residual calibration variants for one prediction table."""
    if raw_frame.empty:
        return raw_frame
    frames = [raw_frame.copy()]
    val = raw_frame.loc[raw_frame["split"].eq("val")].copy()
    if val.empty:
        return raw_frame

    # Global median residual calibration.
    residual = val[["true_q25", "true_q50", "true_q75"]].to_numpy(float) - val[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(float)
    global_shift = np.nanmedian(residual, axis=0)

    for cal_name, shift in [("global_residual_median_cal", global_shift)]:
        cal = raw_frame.copy()
        pred = cal[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(float) + shift.reshape(1, 3)
        pred = enforce_monotone_numpy(pred)
        cal[["pred_q25", "pred_q50", "pred_q75"]] = pred
        cal["source"] = source
        cal["model"] = model
        cal["checkpoint"] = cal_name
        cal["seed"] = str(seed)
        frames.append(cal)

    # q75-decile residual calibration. Useful for risk-screening diagnostics.
    try:
        bins = pd.qcut(val["pred_q75"], q=10, labels=False, duplicates="drop")
        if bins.notna().nunique() >= 2:
            val = val.assign(_cal_bin=bins.astype("Int64"))
            shifts_by_bin = {}
            for b, group in val.groupby("_cal_bin"):
                res = group[["true_q25", "true_q50", "true_q75"]].to_numpy(float) - group[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(float)
                shifts_by_bin[int(b)] = np.nanmedian(res, axis=0)
            cal = raw_frame.copy()
            pred_bins = pd.qcut(cal["pred_q75"], q=10, labels=False, duplicates="drop")
            pred_arr = cal[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(float)
            for i, b in enumerate(pred_bins):
                if pd.notna(b) and int(b) in shifts_by_bin:
                    pred_arr[i] += shifts_by_bin[int(b)]
                else:
                    pred_arr[i] += global_shift
            cal[["pred_q25", "pred_q50", "pred_q75"]] = enforce_monotone_numpy(pred_arr)
            cal["source"] = source
            cal["model"] = model
            cal["checkpoint"] = "q75_decile_residual_cal"
            cal["seed"] = str(seed)
            frames.append(cal)
    except Exception as exc:
        print(f"Calibration decile variant skipped for {model}: {exc}")

    return pd.concat(frames, ignore_index=True)


def build_leaderboard(metrics: pd.DataFrame, split: str = "test") -> pd.DataFrame:
    """Build a sorted test leaderboard."""
    if metrics.empty:
        return pd.DataFrame()
    board = metrics.loc[metrics["split"].eq(split)].copy()
    if board.empty:
        return board
    return board.sort_values(["pinball_mean", "mae_q75", "iqr_mae"], ascending=True).reset_index(drop=True)


## 9. Neural baseline models

The neural baselines use the same residual monotone quantile head as the graph notebooks. This keeps the comparison fair: the main difference between baselines is the representation family, not the quantile parameterization.


In [11]:
if TORCH_AVAILABLE:
    TAUS_TORCH = torch.tensor([0.25, 0.50, 0.75], dtype=torch.float32)

    def inverse_softplus(x: torch.Tensor, eps: float = 1.0e-6) -> torch.Tensor:
        """Numerically stable inverse softplus for positive tensors."""
        x = torch.clamp(x, min=eps)
        return torch.where(x > 20.0, x, torch.log(torch.expm1(x)))

    def pinball_loss_torch(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """Mean pinball loss per row."""
        error = target - pred
        taus = TAUS_TORCH.to(pred.device).view(1, 3)
        return torch.maximum(taus * error, (taus - 1.0) * error).mean(dim=1)

    def weighted_pinball_loss_torch(pred: torch.Tensor, target: torch.Tensor, weight: torch.Tensor) -> torch.Tensor:
        """Weighted batch pinball loss."""
        return (pinball_loss_torch(pred, target) * weight).mean()

    def iqr_loss_torch(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        """SmoothL1 auxiliary loss on IQR width."""
        return F.smooth_l1_loss(pred[:, 2] - pred[:, 0], target[:, 2] - target[:, 0])

    class FeedForward(nn.Module):
        """Compact GELU feed-forward block."""
        def __init__(self, in_dim: int, hidden_dim: int, out_dim: int, dropout: float):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(in_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, out_dim),
            )

        def forward(self, x: torch.Tensor) -> torch.Tensor:
            return self.net(x)

    class ResidualMonotoneHeadMixin:
        """Mixin implementing the residual monotone quantile head."""
        @staticmethod
        def residual_head(delta: torch.Tensor, base_prior: torch.Tensor) -> torch.Tensor:
            base_q25 = torch.clamp(base_prior[:, 0], min=1.0e-6)
            base_gap50 = torch.clamp(base_prior[:, 1] - base_prior[:, 0], min=1.0e-6)
            base_gap75 = torch.clamp(base_prior[:, 2] - base_prior[:, 1], min=1.0e-6)
            q25 = F.softplus(inverse_softplus(base_q25) + delta[:, 0])
            gap50 = F.softplus(inverse_softplus(base_gap50) + delta[:, 1])
            gap75 = F.softplus(inverse_softplus(base_gap75) + delta[:, 2])
            q50 = q25 + gap50
            q75 = q50 + gap75
            return torch.stack([q25, q50, q75], dim=-1)

    class IDMLPQuantile(nn.Module, ResidualMonotoneHeadMixin):
        """STID-style OD quantile baseline with origin/destination/year identity embeddings."""
        def __init__(self, num_zones: int, num_years: int, numeric_dim: int, embedding_dim: int, hidden_dim: int, dropout: float, use_numeric: bool = True, use_id: bool = True):
            super().__init__()
            self.use_numeric = bool(use_numeric)
            self.use_id = bool(use_id)
            self.numeric_dim = int(numeric_dim)
            if self.use_id:
                self.origin_emb = nn.Embedding(num_zones, embedding_dim)
                self.dest_emb = nn.Embedding(num_zones, embedding_dim)
                self.year_emb = nn.Embedding(num_years, embedding_dim)
            if self.use_numeric and numeric_dim > 0:
                self.numeric_mlp = FeedForward(numeric_dim, hidden_dim, embedding_dim, dropout)
                numeric_out_dim = embedding_dim
            else:
                self.numeric_mlp = None
                numeric_out_dim = 0
            id_dim = embedding_dim * 5 if self.use_id else 0
            decoder_dim = id_dim + numeric_out_dim
            if decoder_dim == 0:
                decoder_dim = 1
                self.constant = nn.Parameter(torch.zeros(1))
            else:
                self.constant = None
            self.decoder = FeedForward(decoder_dim, hidden_dim, hidden_dim, dropout)
            self.out = nn.Linear(hidden_dim, 3)

        def forward(self, batch: Mapping[str, torch.Tensor]) -> torch.Tensor:
            pieces = []
            if self.use_id:
                o = self.origin_emb(batch["origin_idx"])
                d = self.dest_emb(batch["dest_idx"])
                y = self.year_emb(batch["year_idx"])
                pieces.extend([o, d, torch.abs(o - d), o * d, y])
            if self.use_numeric and self.numeric_mlp is not None:
                pieces.append(self.numeric_mlp(batch["x_num"]))
            if pieces:
                z = torch.cat(pieces, dim=-1)
            else:
                z = self.constant.expand(batch["base_prior"].shape[0], 1)
            delta = self.out(self.decoder(z))
            return self.residual_head(delta, batch["base_prior"])

    class ODGCNLiteQuantile(nn.Module, ResidualMonotoneHeadMixin):
        """OD-GCN-lite baseline: homogeneous FAF-zone GCN encoder plus OD edge decoder."""
        def __init__(self, node_dim: int, numeric_dim: int, num_years: int, hidden_dim: int, dropout: float, num_layers: int):
            super().__init__()
            if not PYG_AVAILABLE:
                raise RuntimeError("PyTorch Geometric is required for ODGCNLiteQuantile.")
            self.node_proj = nn.Linear(node_dim, hidden_dim)
            self.convs = nn.ModuleList([GCNConv(hidden_dim, hidden_dim) for _ in range(num_layers)])
            self.edge_mlp = FeedForward(numeric_dim, hidden_dim, hidden_dim, dropout) if numeric_dim > 0 else None
            self.year_emb = nn.Embedding(num_years, hidden_dim)
            edge_dim = hidden_dim if numeric_dim > 0 else 0
            decoder_dim = hidden_dim * 5 + edge_dim
            self.decoder = FeedForward(decoder_dim, hidden_dim, hidden_dim, dropout)
            self.out = nn.Linear(hidden_dim, 3)

        def encode(self, node_x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
            h = F.gelu(self.node_proj(node_x))
            for conv in self.convs:
                h = F.gelu(conv(h, edge_index))
            return h

        def forward(self, batch: Mapping[str, torch.Tensor], node_x: torch.Tensor, graph_edge_index: torch.Tensor) -> torch.Tensor:
            h = self.encode(node_x, graph_edge_index)
            o = h[batch["origin_idx"]]
            d = h[batch["dest_idx"]]
            y = self.year_emb(batch["year_idx"])
            pieces = [o, d, torch.abs(o - d), o * d, y]
            if self.edge_mlp is not None:
                pieces.append(self.edge_mlp(batch["x_num"]))
            z = torch.cat(pieces, dim=-1)
            delta = self.out(self.decoder(z))
            return self.residual_head(delta, batch["base_prior"])

    class RGCNQuantile(nn.Module, ResidualMonotoneHeadMixin):
        """Relation-aware R-GCN baseline over the typed freight graph flattened to relation IDs."""
        def __init__(self, node_dim: int, numeric_dim: int, num_years: int, num_relations: int, hidden_dim: int, dropout: float, num_layers: int):
            super().__init__()
            if not PYG_AVAILABLE:
                raise RuntimeError("PyTorch Geometric is required for RGCNQuantile.")
            self.node_proj = nn.Linear(node_dim, hidden_dim)
            self.convs = nn.ModuleList([
                RGCNConv(hidden_dim, hidden_dim, num_relations=num_relations, num_bases=min(8, max(1, num_relations)))
                for _ in range(num_layers)
            ])
            self.edge_mlp = FeedForward(numeric_dim, hidden_dim, hidden_dim, dropout) if numeric_dim > 0 else None
            self.year_emb = nn.Embedding(num_years, hidden_dim)
            edge_dim = hidden_dim if numeric_dim > 0 else 0
            decoder_dim = hidden_dim * 5 + edge_dim
            self.decoder = FeedForward(decoder_dim, hidden_dim, hidden_dim, dropout)
            self.out = nn.Linear(hidden_dim, 3)

        def encode(self, node_x: torch.Tensor, edge_index: torch.Tensor, edge_type: torch.Tensor) -> torch.Tensor:
            h = F.gelu(self.node_proj(node_x))
            for conv in self.convs:
                h = F.gelu(conv(h, edge_index, edge_type))
            return h

        def forward(self, batch: Mapping[str, torch.Tensor], node_x: torch.Tensor, graph_edge_index: torch.Tensor, graph_edge_type: torch.Tensor, supervised_edge_index_global: torch.Tensor) -> torch.Tensor:
            h = self.encode(node_x, graph_edge_index, graph_edge_type)
            src = supervised_edge_index_global[0, batch["row_idx"]]
            dst = supervised_edge_index_global[1, batch["row_idx"]]
            o = h[src]
            d = h[dst]
            y = self.year_emb(batch["year_idx"])
            pieces = [o, d, torch.abs(o - d), o * d, y]
            if self.edge_mlp is not None:
                pieces.append(self.edge_mlp(batch["x_num"]))
            z = torch.cat(pieces, dim=-1)
            delta = self.out(self.decoder(z))
            return self.residual_head(delta, batch["base_prior"])


## 10. Neural training, checkpoint resume, and prediction

Each `(model, seed)` run has a directory containing:

- `last.pt` — most recent epoch, optimizer state, best metrics, and patience counter.
- `best_val_pinball.pt`, `best_val_q75.pt`, `best_val_iqr.pt` — validation-selected checkpoints.
- `history.csv` — epoch-level training and validation metrics.
- `done.json` — completion marker.

If a run is interrupted, rerunning the notebook resumes from `last.pt`.


In [12]:
if TORCH_AVAILABLE:
    class NeuralDataBundle:
        """Container for common tensors used by neural baselines."""
        def __init__(self, x_num: np.ndarray):
            self.x_num_np = np.asarray(x_num, dtype=np.float32)
            self.device = torch.device(cfg.device)
            self.x_num = torch.tensor(self.x_num_np, dtype=torch.float32, device=self.device)
            self.y = torch.tensor(y_scaled_np, dtype=torch.float32, device=self.device)
            self.base_prior = torch.tensor(base_prior_scaled_np, dtype=torch.float32, device=self.device)
            self.weight = torch.tensor(sample_weight_np, dtype=torch.float32, device=self.device)
            self.origin_idx = torch.tensor(origin_idx_np, dtype=torch.long, device=self.device)
            self.dest_idx = torch.tensor(dest_idx_np, dtype=torch.long, device=self.device)
            self.year_idx = torch.tensor(year_idx_np, dtype=torch.long, device=self.device)
            self.row_idx = torch.arange(len(supervised_df), dtype=torch.long, device=self.device)

        def make_batch(self, indices: torch.Tensor) -> Dict[str, torch.Tensor]:
            return {
                "row_idx": self.row_idx[indices],
                "x_num": self.x_num[indices],
                "y": self.y[indices],
                "base_prior": self.base_prior[indices],
                "weight": self.weight[indices],
                "origin_idx": self.origin_idx[indices],
                "dest_idx": self.dest_idx[indices],
                "year_idx": self.year_idx[indices],
            }


def make_index_loader(indices: np.ndarray, batch_size: int, shuffle: bool, seed: int) -> DataLoader:
    """Create a deterministic DataLoader over row indices."""
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    tensor = torch.tensor(np.asarray(indices, dtype=np.int64), dtype=torch.long)
    return DataLoader(TensorDataset(tensor), batch_size=batch_size, shuffle=shuffle, generator=generator)


def validation_metrics_torch(pred_scaled: np.ndarray, true_scaled: np.ndarray) -> Dict[str, float]:
    """Compute validation selection metrics on scaled values."""
    pred = torch.tensor(pred_scaled, dtype=torch.float32)
    target = torch.tensor(true_scaled, dtype=torch.float32)
    loss = pinball_loss_torch(pred, target).mean().item()
    q75 = torch.mean(torch.abs(pred[:, 2] - target[:, 2])).item()
    iqr = torch.mean(torch.abs((pred[:, 2] - pred[:, 0]) - (target[:, 2] - target[:, 0]))).item()
    return {"val_pinball": loss, "val_q75_mae": q75, "val_iqr_mae": iqr}


def predict_neural_scaled(
    model: nn.Module,
    bundle: "NeuralDataBundle",
    indices: np.ndarray,
    extra: Optional[Mapping[str, torch.Tensor]] = None,
    batch_size: Optional[int] = None,
) -> np.ndarray:
    """Predict scaled quantiles for a neural baseline."""
    model.eval()
    preds = []
    extra = extra or {}
    loader = make_index_loader(indices, batch_size or cfg.batch_size, shuffle=False, seed=0)
    with torch.no_grad():
        for (idx_cpu,) in loader:
            idx = idx_cpu.to(bundle.device)
            batch = bundle.make_batch(idx)
            if isinstance(model, ODGCNLiteQuantile):
                pred = model(batch, extra["node_x"], extra["edge_index"])
            elif isinstance(model, RGCNQuantile):
                pred = model(batch, extra["node_x"], extra["edge_index"], extra["edge_type"], extra["supervised_edge_index_global"])
            else:
                pred = model(batch)
            preds.append(pred.detach().cpu().numpy())
    return np.vstack(preds).astype(np.float32)


def save_torch_checkpoint(payload: Mapping[str, Any], path: Path) -> None:
    """Atomically save a PyTorch checkpoint."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(dict(payload), tmp)
    tmp.replace(path)


def train_neural_baseline(
    model_name: str,
    seed: int,
    model_factory: Callable[[], nn.Module],
    bundle: "NeuralDataBundle",
    extra: Optional[Mapping[str, torch.Tensor]] = None,
) -> Tuple[Dict[str, Path], pd.DataFrame]:
    """Train one neural baseline for one seed with checkpoint resume."""
    set_global_seed(seed)
    extra = dict(extra or {})
    model_dir = paths.models_dir / model_name / f"seed_{seed}"
    model_dir.mkdir(parents=True, exist_ok=True)
    last_path = model_dir / "last.pt"
    done_path = model_dir / "done.json"
    history_path = model_dir / "history.csv"
    best_paths = {
        "best_val_pinball": model_dir / "best_val_pinball.pt",
        "best_val_q75": model_dir / "best_val_q75.pt",
        "best_val_iqr": model_dir / "best_val_iqr.pt",
    }

    if cfg.resume and done_path.exists() and all(p.exists() for p in best_paths.values()):
        history = pd.read_csv(history_path) if history_path.exists() else pd.DataFrame()
        print(f"[resume] {model_name} seed={seed} already completed. Reusing checkpoints.")
        return best_paths, history

    model = model_factory().to(bundle.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    start_epoch = 1
    best_values = {"best_val_pinball": math.inf, "best_val_q75": math.inf, "best_val_iqr": math.inf}
    best_epochs = {key: -1 for key in best_values}
    epochs_since_best = 0
    history_rows: List[Dict[str, Any]] = []

    if cfg.resume and last_path.exists():
        ckpt = torch.load(last_path, map_location=bundle.device)
        model.load_state_dict(ckpt["model_state"])
        optimizer.load_state_dict(ckpt["optimizer_state"])
        start_epoch = int(ckpt.get("epoch", 0)) + 1
        best_values.update(ckpt.get("best_values", {}))
        best_epochs.update(ckpt.get("best_epochs", {}))
        epochs_since_best = int(ckpt.get("epochs_since_best", 0))
        history_rows = list(ckpt.get("history_rows", []))
        print(f"[resume] {model_name} seed={seed} from epoch {start_epoch}")

    train_loader = make_index_loader(split_indices_np["train"], cfg.batch_size, shuffle=True, seed=seed)
    progress = ProgressPrinter(cfg.progress_min_interval_sec)

    for epoch in range(start_epoch, cfg.max_epochs + 1):
        model.train()
        batch_losses = []
        n_batches = len(train_loader)
        for batch_no, (idx_cpu,) in enumerate(train_loader, start=1):
            idx = idx_cpu.to(bundle.device)
            batch = bundle.make_batch(idx)
            if isinstance(model, ODGCNLiteQuantile):
                pred = model(batch, extra["node_x"], extra["edge_index"])
            elif isinstance(model, RGCNQuantile):
                pred = model(batch, extra["node_x"], extra["edge_index"], extra["edge_type"], extra["supervised_edge_index_global"])
            else:
                pred = model(batch)
            target = batch["y"]
            weight = batch["weight"]
            loss = weighted_pinball_loss_torch(pred, target, weight)
            if cfg.lambda_iqr > 0:
                loss = loss + cfg.lambda_iqr * iqr_loss_torch(pred, target)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            if cfg.grad_clip_norm > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)
            optimizer.step()
            batch_losses.append(float(loss.detach().cpu().item()))
            progress.update(f"[{model_name} seed={seed}] epoch {epoch:03d}/{cfg.max_epochs} batch {batch_no:03d}/{n_batches:03d} loss={batch_losses[-1]:.5f}")

        val_pred = predict_neural_scaled(model, bundle, split_indices_np["val"], extra=extra)
        val_metrics = validation_metrics_torch(val_pred, y_scaled_np[mask_dict["val"]])
        train_loss = float(np.mean(batch_losses)) if batch_losses else np.nan
        history_row = {"model": model_name, "seed": seed, "epoch": epoch, "train_loss": train_loss, **val_metrics}
        history_rows.append(history_row)

        improved = False
        candidates = {
            "best_val_pinball": val_metrics["val_pinball"],
            "best_val_q75": val_metrics["val_q75_mae"],
            "best_val_iqr": val_metrics["val_iqr_mae"],
        }
        for key, value in candidates.items():
            if float(value) < float(best_values[key]):
                best_values[key] = float(value)
                best_epochs[key] = int(epoch)
                save_torch_checkpoint({
                    "model_state": model.state_dict(),
                    "epoch": epoch,
                    "selection_metric": key,
                    "selection_value": float(value),
                    "val_metrics": val_metrics,
                    "model_name": model_name,
                    "seed": seed,
                }, best_paths[key])
                improved = True

        epochs_since_best = 0 if improved else epochs_since_best + 1
        save_torch_checkpoint({
            "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "epoch": epoch,
            "best_values": best_values,
            "best_epochs": best_epochs,
            "epochs_since_best": epochs_since_best,
            "history_rows": history_rows,
            "model_name": model_name,
            "seed": seed,
        }, last_path)
        pd.DataFrame(history_rows).to_csv(history_path, index=False)
        progress.update(
            f"[{model_name} seed={seed}] epoch {epoch:03d}/{cfg.max_epochs} "
            f"train={train_loss:.5f} val_pin={val_metrics['val_pinball']:.5f} "
            f"val_q75={val_metrics['val_q75_mae']:.5f} val_iqr={val_metrics['val_iqr_mae']:.5f} "
            f"patience={epochs_since_best}/{cfg.patience}",
            force=True,
        )

        if epochs_since_best >= cfg.patience:
            progress.done(f"[{model_name} seed={seed}] early stopped at epoch {epoch}")
            break

    progress.done(f"[{model_name} seed={seed}] completed")
    atomic_write_json({"model": model_name, "seed": seed, "best_values": best_values, "best_epochs": best_epochs, "completed_at": time.time()}, done_path)
    return best_paths, pd.DataFrame(history_rows)


def predict_neural_checkpoints(
    model_name: str,
    seed: int,
    model_factory: Callable[[], nn.Module],
    bundle: "NeuralDataBundle",
    checkpoint_paths: Mapping[str, Path],
    extra: Optional[Mapping[str, torch.Tensor]] = None,
) -> pd.DataFrame:
    """Load best checkpoints and return validation/test prediction frames."""
    frames = []
    for checkpoint_name, ckpt_path in checkpoint_paths.items():
        if not ckpt_path.exists():
            print(f"Warning: missing checkpoint {ckpt_path}")
            continue
        model = model_factory().to(bundle.device)
        ckpt = torch.load(ckpt_path, map_location=bundle.device)
        model.load_state_dict(ckpt["model_state"])
        for split_name in ["val", "test"]:
            idx = split_indices_np[split_name]
            pred_scaled = predict_neural_scaled(model, bundle, idx, extra=extra)
            pred = pred_scaled * target_scale
            frames.append(make_prediction_frame(idx, pred, source="StrategicBaselines", model=model_name, checkpoint=checkpoint_name, seed=seed))
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


## 11. Run neural strategic baselines

This cell executes the neural baselines selected in the configuration. It automatically skips graph baselines when graph artifacts or PyG are unavailable.


In [14]:
# =============================================================================
# Neural training, robust checkpoint resume, and strategic baseline execution
# =============================================================================
#
# This cell replaces the original:
#   10. Neural training, checkpoint resume, and prediction
#   11. Run neural strategic baselines
#
# Main bug fixed:
#   On Windows, Path.replace() can fail with PermissionError if PyCharm,
#   Windows Defender, or another process briefly locks last.pt / best_*.pt.
#
# This implementation uses:
#   - unique temporary checkpoint files,
#   - file-handle flush + fsync,
#   - os.replace with retry/backoff,
#   - emergency fallback checkpoint names,
#   - explicit torch.load(..., weights_only=...) handling,
#   - stale .tmp cleanup before each model/seed run.
# =============================================================================

import contextlib
import gc
import json
import math
import os
import time
import uuid
import warnings
from pathlib import Path
from typing import Any, Callable, Dict, List, Mapping, Optional, Tuple

import numpy as np
import pandas as pd

if TORCH_AVAILABLE:
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset

    # -------------------------------------------------------------------------
    # 0. Runtime device guard
    # -------------------------------------------------------------------------
    # If cfg.device is "cuda" but the current kernel cannot see CUDA, fall back
    # to CPU instead of crashing during tensor construction.
    if str(cfg.device).startswith("cuda") and not torch.cuda.is_available():
        print(
            "[device] cfg.device requests CUDA, but torch.cuda.is_available() "
            "is False in this kernel. Falling back to CPU for this cell."
        )
        cfg.device = "cpu"

    # -------------------------------------------------------------------------
    # 1. Data bundle
    # -------------------------------------------------------------------------
    class NeuralDataBundle:
        """Container for common tensors used by neural baselines."""

        def __init__(self, x_num: np.ndarray):
            self.x_num_np = np.asarray(x_num, dtype=np.float32)
            self.device = torch.device(cfg.device)

            self.x_num = torch.tensor(self.x_num_np, dtype=torch.float32, device=self.device)
            self.y = torch.tensor(y_scaled_np, dtype=torch.float32, device=self.device)
            self.base_prior = torch.tensor(base_prior_scaled_np, dtype=torch.float32, device=self.device)
            self.weight = torch.tensor(sample_weight_np, dtype=torch.float32, device=self.device)

            self.origin_idx = torch.tensor(origin_idx_np, dtype=torch.long, device=self.device)
            self.dest_idx = torch.tensor(dest_idx_np, dtype=torch.long, device=self.device)
            self.year_idx = torch.tensor(year_idx_np, dtype=torch.long, device=self.device)

            # row_idx is the global supervised-row index. R-GCN uses it to map
            # supervised OD rows back to flattened global graph edge endpoints.
            self.row_idx = torch.arange(len(supervised_df), dtype=torch.long, device=self.device)

        def make_batch(self, indices: torch.Tensor) -> Dict[str, torch.Tensor]:
            """Create a mini-batch dictionary from global supervised-row indices."""
            return {
                "row_idx": self.row_idx[indices],
                "x_num": self.x_num[indices],
                "y": self.y[indices],
                "base_prior": self.base_prior[indices],
                "weight": self.weight[indices],
                "origin_idx": self.origin_idx[indices],
                "dest_idx": self.dest_idx[indices],
                "year_idx": self.year_idx[indices],
            }

    # -------------------------------------------------------------------------
    # 2. DataLoader and metric helpers
    # -------------------------------------------------------------------------
    def make_index_loader(
        indices: np.ndarray,
        batch_size: int,
        shuffle: bool,
        seed: int,
    ) -> DataLoader:
        """Create a deterministic DataLoader over row indices."""
        generator = torch.Generator()
        generator.manual_seed(int(seed))

        indices_np = np.asarray(indices, dtype=np.int64)
        tensor = torch.tensor(indices_np, dtype=torch.long)

        return DataLoader(
            TensorDataset(tensor),
            batch_size=int(batch_size),
            shuffle=bool(shuffle),
            generator=generator,
            drop_last=False,
        )

    def validation_metrics_torch(
        pred_scaled: np.ndarray,
        true_scaled: np.ndarray,
    ) -> Dict[str, float]:
        """Compute validation selection metrics on scaled target values."""
        pred = torch.tensor(pred_scaled, dtype=torch.float32)
        target = torch.tensor(true_scaled, dtype=torch.float32)

        loss = pinball_loss_torch(pred, target).mean().item()
        q75 = torch.mean(torch.abs(pred[:, 2] - target[:, 2])).item()
        iqr = torch.mean(
            torch.abs((pred[:, 2] - pred[:, 0]) - (target[:, 2] - target[:, 0]))
        ).item()

        return {
            "val_pinball": float(loss),
            "val_q75_mae": float(q75),
            "val_iqr_mae": float(iqr),
        }

    def predict_neural_scaled(
        model: nn.Module,
        bundle: NeuralDataBundle,
        indices: np.ndarray,
        extra: Optional[Mapping[str, torch.Tensor]] = None,
        batch_size: Optional[int] = None,
    ) -> np.ndarray:
        """Predict scaled quantiles for a neural baseline."""
        model.eval()
        preds: List[np.ndarray] = []
        extra = dict(extra or {})

        loader = make_index_loader(
            indices=np.asarray(indices, dtype=np.int64),
            batch_size=batch_size or cfg.batch_size,
            shuffle=False,
            seed=0,
        )

        with torch.no_grad():
            for (idx_cpu,) in loader:
                idx = idx_cpu.to(bundle.device)
                batch = bundle.make_batch(idx)

                if isinstance(model, ODGCNLiteQuantile):
                    pred = model(batch, extra["node_x"], extra["edge_index"])
                elif isinstance(model, RGCNQuantile):
                    pred = model(
                        batch,
                        extra["node_x"],
                        extra["edge_index"],
                        extra["edge_type"],
                        extra["supervised_edge_index_global"],
                    )
                else:
                    pred = model(batch)

                preds.append(pred.detach().cpu().numpy())

        if not preds:
            return np.empty((0, 3), dtype=np.float32)

        return np.vstack(preds).astype(np.float32)

    # -------------------------------------------------------------------------
    # 3. Robust checkpoint I/O
    # -------------------------------------------------------------------------
    def _checkpoint_tmp_patterns(path: Path) -> List[str]:
        """Return temporary-file glob patterns associated with one checkpoint path."""
        return [
            f"{path.name}.tmp",
            f"{path.name}.tmp.*",
            f".{path.name}.tmp.*",
        ]

    def cleanup_checkpoint_tmp_files(path_or_dir: Path) -> None:
        """Remove stale temporary checkpoint files.

        This is intentionally best-effort. If Windows is still locking a temp
        file, we skip it rather than failing the run.
        """
        path_or_dir = Path(path_or_dir)

        if path_or_dir.is_dir():
            candidates = list(path_or_dir.glob("*.tmp")) + list(path_or_dir.glob("*.tmp.*"))
        else:
            candidates = []
            for pattern in _checkpoint_tmp_patterns(path_or_dir):
                candidates.extend(path_or_dir.parent.glob(pattern))

        for tmp_path in candidates:
            with contextlib.suppress(Exception):
                tmp_path.unlink()

    def _fsync_torch_save(payload: Mapping[str, Any], tmp_path: Path) -> None:
        """Save a checkpoint to a temp file and force the file handle to close."""
        tmp_path.parent.mkdir(parents=True, exist_ok=True)

        with open(tmp_path, "wb") as f:
            torch.save(dict(payload), f)
            f.flush()
            os.fsync(f.fileno())

    def save_torch_checkpoint(
        payload: Mapping[str, Any],
        path: Path,
        *,
        max_retries: int = 40,
        base_sleep_sec: float = 0.10,
    ) -> Path:
        """Safely save a PyTorch checkpoint on Windows.

        Returns
        -------
        Path
            The checkpoint path that was successfully written. Usually this is
            `path`. If Windows keeps the target locked after all retries, the
            function writes an emergency sidecar checkpoint and returns that path.

        Notes
        -----
        The old implementation used:
            tmp = path.with_suffix(path.suffix + ".tmp")
            torch.save(..., tmp)
            tmp.replace(path)

        On Windows this can fail with PermissionError if the target is briefly
        locked. This implementation uses a unique temporary file, retries
        os.replace(), and falls back gracefully.
        """
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        # Unique temp file avoids collisions with stale last.pt.tmp from a
        # previous interrupted notebook run.
        tmp_path = path.with_name(f".{path.name}.tmp.{os.getpid()}.{uuid.uuid4().hex}")

        # Remove stale temp files for this specific checkpoint before writing.
        cleanup_checkpoint_tmp_files(path)

        _fsync_torch_save(payload, tmp_path)

        last_error: Optional[BaseException] = None

        for attempt in range(int(max_retries)):
            try:
                os.replace(str(tmp_path), str(path))
                return path
            except PermissionError as exc:
                last_error = exc

                # Release any Python-side references and CUDA cache. This does
                # not unlock external processes, but helps when the current
                # process is holding tensors longer than necessary.
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                sleep_sec = base_sleep_sec * min(10.0, 1.0 + attempt * 0.35)
                time.sleep(sleep_sec)

            except OSError as exc:
                last_error = exc
                time.sleep(base_sleep_sec)

        # Fallback 1: try a direct write. This is not atomic, but better than
        # killing a long training run when the target was only briefly blocked.
        try:
            _fsync_torch_save(payload, path)
            with contextlib.suppress(Exception):
                tmp_path.unlink()
            return path
        except Exception as direct_exc:
            last_error = direct_exc

        # Fallback 2: save an emergency sidecar checkpoint. The caller can use
        # the returned path for best checkpoint dictionaries.
        emergency_path = path.with_name(
            f"{path.stem}.emergency_{int(time.time())}_{uuid.uuid4().hex[:8]}{path.suffix}"
        )

        try:
            os.replace(str(tmp_path), str(emergency_path))
            warnings.warn(
                f"Could not update checkpoint target after retries: {path}. "
                f"Saved emergency checkpoint instead: {emergency_path}. "
                f"Last error: {last_error}",
                RuntimeWarning,
            )
            return emergency_path
        except Exception:
            with contextlib.suppress(Exception):
                tmp_path.unlink()
            raise RuntimeError(
                f"Failed to save checkpoint to {path}. Last error: {last_error}"
            ) from last_error

    def load_torch_checkpoint(path: Path, map_location: Any) -> Dict[str, Any]:
        """Load a local project checkpoint without torch.load warning spam."""
        path = Path(path)

        # Prefer weights_only=True when supported. Our checkpoints contain
        # tensors plus primitive dict/list/float/int/string metadata.
        try:
            return torch.load(path, map_location=map_location, weights_only=True)
        except TypeError:
            # Older PyTorch versions do not support weights_only.
            return torch.load(path, map_location=map_location)
        except Exception:
            # Some optimizer or history metadata can fail under weights_only on
            # certain PyTorch versions. These are local checkpoints generated by
            # this notebook, so explicit weights_only=False is acceptable.
            return torch.load(path, map_location=map_location, weights_only=False)

    def safe_write_json(payload: Mapping[str, Any], path: Path) -> None:
        """Write a JSON file robustly using the same retry logic as checkpoints."""
        path = Path(path)
        path.parent.mkdir(parents=True, exist_ok=True)

        tmp_path = path.with_name(f".{path.name}.tmp.{os.getpid()}.{uuid.uuid4().hex}")
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(dict(payload), f, indent=2, sort_keys=True)
            f.flush()
            os.fsync(f.fileno())

        last_error: Optional[BaseException] = None
        for attempt in range(30):
            try:
                os.replace(str(tmp_path), str(path))
                return
            except PermissionError as exc:
                last_error = exc
                time.sleep(0.10 * min(10.0, 1.0 + attempt * 0.35))

        try:
            with open(path, "w", encoding="utf-8") as f:
                json.dump(dict(payload), f, indent=2, sort_keys=True)
        finally:
            with contextlib.suppress(Exception):
                tmp_path.unlink()

        if last_error is not None:
            warnings.warn(f"JSON atomic replace failed for {path}; direct write was used.")

    # -------------------------------------------------------------------------
    # 4. Training and prediction functions
    # -------------------------------------------------------------------------
    def train_neural_baseline(
        model_name: str,
        seed: int,
        model_factory: Callable[[], nn.Module],
        bundle: NeuralDataBundle,
        extra: Optional[Mapping[str, torch.Tensor]] = None,
    ) -> Tuple[Dict[str, Path], pd.DataFrame]:
        """Train one neural baseline for one seed with checkpoint resume."""
        set_global_seed(seed)
        extra = dict(extra or {})

        model_dir = paths.models_dir / model_name / f"seed_{seed}"
        model_dir.mkdir(parents=True, exist_ok=True)

        last_path = model_dir / "last.pt"
        done_path = model_dir / "done.json"
        history_path = model_dir / "history.csv"

        best_paths: Dict[str, Path] = {
            "best_val_pinball": model_dir / "best_val_pinball.pt",
            "best_val_q75": model_dir / "best_val_q75.pt",
            "best_val_iqr": model_dir / "best_val_iqr.pt",
        }

        cleanup_checkpoint_tmp_files(model_dir)

        if cfg.resume and done_path.exists() and all(p.exists() for p in best_paths.values()):
            history = pd.read_csv(history_path) if history_path.exists() else pd.DataFrame()
            print(f"[resume] {model_name} seed={seed} already completed. Reusing checkpoints.")
            return best_paths, history

        model = model_factory().to(bundle.device)
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cfg.learning_rate,
            weight_decay=cfg.weight_decay,
        )

        start_epoch = 1
        best_values: Dict[str, float] = {
            "best_val_pinball": math.inf,
            "best_val_q75": math.inf,
            "best_val_iqr": math.inf,
        }
        best_epochs: Dict[str, int] = {key: -1 for key in best_values}
        epochs_since_best = 0
        history_rows: List[Dict[str, Any]] = []

        if cfg.resume and last_path.exists():
            try:
                ckpt = load_torch_checkpoint(last_path, map_location=bundle.device)
                model.load_state_dict(ckpt["model_state"])
                optimizer.load_state_dict(ckpt["optimizer_state"])

                start_epoch = int(ckpt.get("epoch", 0)) + 1
                best_values.update({k: float(v) for k, v in ckpt.get("best_values", {}).items()})
                best_epochs.update({k: int(v) for k, v in ckpt.get("best_epochs", {}).items()})
                epochs_since_best = int(ckpt.get("epochs_since_best", 0))
                history_rows = list(ckpt.get("history_rows", []))

                print(f"[resume] {model_name} seed={seed} from epoch {start_epoch}")

                del ckpt
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

            except Exception as exc:
                warnings.warn(
                    f"Could not resume from {last_path}. "
                    f"Starting {model_name} seed={seed} from scratch. Error: {exc}",
                    RuntimeWarning,
                )
                start_epoch = 1
                best_values = {
                    "best_val_pinball": math.inf,
                    "best_val_q75": math.inf,
                    "best_val_iqr": math.inf,
                }
                best_epochs = {key: -1 for key in best_values}
                epochs_since_best = 0
                history_rows = []

        train_loader = make_index_loader(
            indices=split_indices_np["train"],
            batch_size=cfg.batch_size,
            shuffle=True,
            seed=seed,
        )

        progress = ProgressPrinter(cfg.progress_min_interval_sec)

        for epoch in range(start_epoch, cfg.max_epochs + 1):
            model.train()
            batch_losses: List[float] = []
            n_batches = len(train_loader)

            for batch_no, (idx_cpu,) in enumerate(train_loader, start=1):
                idx = idx_cpu.to(bundle.device)
                batch = bundle.make_batch(idx)

                if isinstance(model, ODGCNLiteQuantile):
                    pred = model(batch, extra["node_x"], extra["edge_index"])
                elif isinstance(model, RGCNQuantile):
                    pred = model(
                        batch,
                        extra["node_x"],
                        extra["edge_index"],
                        extra["edge_type"],
                        extra["supervised_edge_index_global"],
                    )
                else:
                    pred = model(batch)

                target = batch["y"]
                weight = batch["weight"]

                loss = weighted_pinball_loss_torch(pred, target, weight)
                if cfg.lambda_iqr > 0:
                    loss = loss + cfg.lambda_iqr * iqr_loss_torch(pred, target)

                optimizer.zero_grad(set_to_none=True)
                loss.backward()

                if cfg.grad_clip_norm > 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip_norm)

                optimizer.step()

                loss_value = float(loss.detach().cpu().item())
                batch_losses.append(loss_value)

                progress.update(
                    f"[{model_name} seed={seed}] "
                    f"epoch {epoch:03d}/{cfg.max_epochs} "
                    f"batch {batch_no:03d}/{n_batches:03d} "
                    f"loss={loss_value:.5f}"
                )

            # Validation.
            val_idx = np.asarray(split_indices_np["val"], dtype=np.int64)
            val_pred = predict_neural_scaled(model, bundle, val_idx, extra=extra)
            val_true = y_scaled_np[val_idx]
            val_metrics = validation_metrics_torch(val_pred, val_true)

            train_loss = float(np.mean(batch_losses)) if batch_losses else np.nan

            history_row = {
                "model": model_name,
                "seed": int(seed),
                "epoch": int(epoch),
                "train_loss": train_loss,
                **val_metrics,
            }
            history_rows.append(history_row)

            improved = False
            candidates = {
                "best_val_pinball": val_metrics["val_pinball"],
                "best_val_q75": val_metrics["val_q75_mae"],
                "best_val_iqr": val_metrics["val_iqr_mae"],
            }

            for key, value in candidates.items():
                value = float(value)
                if value < float(best_values[key]):
                    best_values[key] = value
                    best_epochs[key] = int(epoch)

                    saved_path = save_torch_checkpoint(
                        {
                            "model_state": model.state_dict(),
                            "epoch": int(epoch),
                            "selection_metric": key,
                            "selection_value": value,
                            "val_metrics": val_metrics,
                            "model_name": model_name,
                            "seed": int(seed),
                        },
                        best_paths[key],
                    )

                    # In the rare case where the original checkpoint path was
                    # locked and an emergency path was used, keep the returned
                    # path so prediction can still load it in this same run.
                    best_paths[key] = saved_path
                    improved = True

            epochs_since_best = 0 if improved else epochs_since_best + 1

            save_torch_checkpoint(
                {
                    "model_state": model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "epoch": int(epoch),
                    "best_values": best_values,
                    "best_epochs": best_epochs,
                    "epochs_since_best": int(epochs_since_best),
                    "history_rows": history_rows,
                    "model_name": model_name,
                    "seed": int(seed),
                },
                last_path,
            )

            pd.DataFrame(history_rows).to_csv(history_path, index=False)

            progress.update(
                f"[{model_name} seed={seed}] "
                f"epoch {epoch:03d}/{cfg.max_epochs} "
                f"train={train_loss:.5f} "
                f"val_pin={val_metrics['val_pinball']:.5f} "
                f"val_q75={val_metrics['val_q75_mae']:.5f} "
                f"val_iqr={val_metrics['val_iqr_mae']:.5f} "
                f"patience={epochs_since_best}/{cfg.patience}",
                force=True,
            )

            if epochs_since_best >= cfg.patience:
                progress.done(f"[{model_name} seed={seed}] early stopped at epoch {epoch}")
                break

        progress.done(f"[{model_name} seed={seed}] completed")

        safe_write_json(
            {
                "model": model_name,
                "seed": int(seed),
                "best_values": best_values,
                "best_epochs": best_epochs,
                "completed_at": time.time(),
                "checkpoint_paths": {k: str(v) for k, v in best_paths.items()},
            },
            done_path,
        )

        del model, optimizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        return best_paths, pd.DataFrame(history_rows)

    def predict_neural_checkpoints(
        model_name: str,
        seed: int,
        model_factory: Callable[[], nn.Module],
        bundle: NeuralDataBundle,
        checkpoint_paths: Mapping[str, Path],
        extra: Optional[Mapping[str, torch.Tensor]] = None,
    ) -> pd.DataFrame:
        """Load best checkpoints and return validation/test prediction frames."""
        frames: List[pd.DataFrame] = []
        extra = dict(extra or {})

        for checkpoint_name, ckpt_path in checkpoint_paths.items():
            ckpt_path = Path(ckpt_path)

            if not ckpt_path.exists():
                print(f"Warning: missing checkpoint {ckpt_path}")
                continue

            model = model_factory().to(bundle.device)

            try:
                ckpt = load_torch_checkpoint(ckpt_path, map_location=bundle.device)
                model.load_state_dict(ckpt["model_state"])
            except Exception as exc:
                warnings.warn(
                    f"Skipping checkpoint {ckpt_path} because it could not be loaded: {exc}",
                    RuntimeWarning,
                )
                del model
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                continue

            for split_name in ["val", "test"]:
                idx = np.asarray(split_indices_np[split_name], dtype=np.int64)
                pred_scaled = predict_neural_scaled(model, bundle, idx, extra=extra)
                pred = pred_scaled * target_scale

                frames.append(
                    make_prediction_frame(
                        idx,
                        pred,
                        source="StrategicBaselines",
                        model=model_name,
                        checkpoint=checkpoint_name,
                        seed=seed,
                    )
                )

            del ckpt, model
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

# =============================================================================
# Run neural strategic baselines
# =============================================================================

neural_prediction_frames: List[pd.DataFrame] = []
neural_history_frames: List[pd.DataFrame] = []
skipped_baselines: List[Dict[str, str]] = []

# Preserve order while removing duplicates.
enabled_neural: List[str] = []
all_enabled = list(cfg.enabled_main_baselines) + (
    list(cfg.enabled_appendix_baselines) if cfg.run_appendix_baselines else []
)

for name in all_enabled:
    if name in {"id_mlp_stid", "zone_id_only_mlp", "numeric_mlp_no_id", "od_gcn_lite", "rgcn"}:
        if name not in enabled_neural:
            enabled_neural.append(name)

if not TORCH_AVAILABLE:
    for name in enabled_neural:
        skipped_baselines.append({"baseline": name, "reason": "PyTorch unavailable"})

else:
    # Number of FAF-zone IDs and year IDs used by embedding baselines.
    num_zones = int(max(origin_idx_np.max(), dest_idx_np.max()) + 1)
    num_years = len(year_to_idx)

    # Shared bundles by feature block.
    bundle_base = NeuralDataBundle(X_base_topology if topology_feature_columns else X_base_prior)
    bundle_empty = NeuralDataBundle(X_empty)

    neural_specs: Dict[str, Dict[str, Any]] = {}

    if "id_mlp_stid" in enabled_neural:
        neural_specs["id_mlp_stid"] = {
            "bundle": bundle_base,
            "factory": lambda bundle=bundle_base: IDMLPQuantile(
                num_zones=num_zones,
                num_years=num_years,
                numeric_dim=bundle.x_num_np.shape[1],
                embedding_dim=cfg.embedding_dim,
                hidden_dim=cfg.hidden_dim,
                dropout=cfg.dropout,
                use_numeric=True,
                use_id=True,
            ),
            "extra": {},
        }

    if "zone_id_only_mlp" in enabled_neural:
        neural_specs["zone_id_only_mlp"] = {
            "bundle": bundle_empty,
            "factory": lambda bundle=bundle_empty: IDMLPQuantile(
                num_zones=num_zones,
                num_years=num_years,
                numeric_dim=0,
                embedding_dim=cfg.embedding_dim,
                hidden_dim=cfg.hidden_dim,
                dropout=cfg.dropout,
                use_numeric=False,
                use_id=True,
            ),
            "extra": {},
        }

    if "numeric_mlp_no_id" in enabled_neural:
        neural_specs["numeric_mlp_no_id"] = {
            "bundle": bundle_base,
            "factory": lambda bundle=bundle_base: IDMLPQuantile(
                num_zones=num_zones,
                num_years=num_years,
                numeric_dim=bundle.x_num_np.shape[1],
                embedding_dim=cfg.embedding_dim,
                hidden_dim=cfg.hidden_dim,
                dropout=cfg.dropout,
                use_numeric=True,
                use_id=False,
            ),
            "extra": {},
        }

    if "od_gcn_lite" in enabled_neural:
        if not PYG_AVAILABLE or faf_node_features_np is None or hom_faf_edge_index_np is None:
            skipped_baselines.append(
                {
                    "baseline": "od_gcn_lite",
                    "reason": "PyG, FAF node features, or homogeneous graph edge_index unavailable",
                }
            )
        else:
            device = torch.device(cfg.device)
            node_x_t = torch.tensor(faf_node_features_np, dtype=torch.float32, device=device)
            edge_index_t = torch.tensor(hom_faf_edge_index_np, dtype=torch.long, device=device)

            neural_specs["od_gcn_lite"] = {
                "bundle": bundle_base,
                "factory": lambda bundle=bundle_base: ODGCNLiteQuantile(
                    node_dim=faf_node_features_np.shape[1],
                    numeric_dim=bundle.x_num_np.shape[1],
                    num_years=num_years,
                    hidden_dim=cfg.hidden_dim,
                    dropout=cfg.dropout,
                    num_layers=cfg.graph_layers,
                ),
                "extra": {
                    "node_x": node_x_t,
                    "edge_index": edge_index_t,
                },
            }

    if "rgcn" in enabled_neural:
        if not PYG_AVAILABLE or rgcn_graph is None:
            skipped_baselines.append(
                {
                    "baseline": "rgcn",
                    "reason": "PyG or flattened R-GCN graph unavailable",
                }
            )
        else:
            device = torch.device(cfg.device)

            neural_specs["rgcn"] = {
                "bundle": bundle_base,
                "factory": lambda bundle=bundle_base: RGCNQuantile(
                    node_dim=rgcn_graph["x_all"].shape[1],
                    numeric_dim=bundle.x_num_np.shape[1],
                    num_years=num_years,
                    num_relations=int(rgcn_graph["num_relations"]),
                    hidden_dim=cfg.hidden_dim,
                    dropout=cfg.dropout,
                    num_layers=cfg.graph_layers,
                ),
                "extra": {
                    "node_x": torch.tensor(rgcn_graph["x_all"], dtype=torch.float32, device=device),
                    "edge_index": torch.tensor(rgcn_graph["edge_index"], dtype=torch.long, device=device),
                    "edge_type": torch.tensor(rgcn_graph["edge_type"], dtype=torch.long, device=device),
                    "supervised_edge_index_global": torch.tensor(
                        rgcn_graph["supervised_edge_index_global"],
                        dtype=torch.long,
                        device=device,
                    ),
                },
            }

    for model_name, spec in neural_specs.items():
        for seed in cfg.seeds:
            print("\n" + "=" * 90)
            print(f"Training neural baseline: {model_name} | seed={seed}")
            print("=" * 90)

            # Clean stale temp files from previous interrupted runs before this
            # model/seed starts. This is important on Windows.
            model_dir = paths.models_dir / model_name / f"seed_{int(seed)}"
            cleanup_checkpoint_tmp_files(model_dir)

            try:
                checkpoint_paths, history = train_neural_baseline(
                    model_name=model_name,
                    seed=int(seed),
                    model_factory=spec["factory"],
                    bundle=spec["bundle"],
                    extra=spec.get("extra", {}),
                )

                if history is not None and not history.empty:
                    neural_history_frames.append(history)

                pred_frame = predict_neural_checkpoints(
                    model_name=model_name,
                    seed=int(seed),
                    model_factory=spec["factory"],
                    bundle=spec["bundle"],
                    checkpoint_paths=checkpoint_paths,
                    extra=spec.get("extra", {}),
                )

                if pred_frame is not None and not pred_frame.empty:
                    neural_prediction_frames.append(pred_frame)

            except PermissionError as exc:
                # This should be rare after the robust saver, but keeping the
                # error explicit helps diagnose external locks.
                skipped_baselines.append(
                    {
                        "baseline": model_name,
                        "seed": str(seed),
                        "reason": f"Windows PermissionError during checkpoint I/O: {exc}",
                    }
                )
                print(f"[skip] {model_name} seed={seed}: {exc}")

            except RuntimeError as exc:
                skipped_baselines.append(
                    {
                        "baseline": model_name,
                        "seed": str(seed),
                        "reason": f"RuntimeError: {exc}",
                    }
                )
                print(f"[skip] {model_name} seed={seed}: {exc}")

            finally:
                gc.collect()
                if TORCH_AVAILABLE and torch.cuda.is_available():
                    torch.cuda.empty_cache()

neural_predictions = (
    pd.concat(neural_prediction_frames, ignore_index=True)
    if neural_prediction_frames
    else pd.DataFrame()
)

neural_history = (
    pd.concat(neural_history_frames, ignore_index=True)
    if neural_history_frames
    else pd.DataFrame()
)

print("Neural predictions:", neural_predictions.shape)
print("Neural history:", neural_history.shape)

if skipped_baselines:
    print("Skipped baselines:")
    print(pd.DataFrame(skipped_baselines).to_string(index=False))


Training neural baseline: id_mlp_stid | seed=7
[resume] id_mlp_stid seed=7 already completed. Reusing checkpoints.

Training neural baseline: id_mlp_stid | seed=42
[resume] id_mlp_stid seed=42 already completed. Reusing checkpoints.

Training neural baseline: id_mlp_stid | seed=123
[resume] id_mlp_stid seed=123 already completed. Reusing checkpoints.

Training neural baseline: id_mlp_stid | seed=2026
[resume] id_mlp_stid seed=2026 already completed. Reusing checkpoints.

Training neural baseline: id_mlp_stid | seed=535
[resume] id_mlp_stid seed=535 already completed. Reusing checkpoints.

Training neural baseline: zone_id_only_mlp | seed=7
[resume] zone_id_only_mlp seed=7 already completed. Reusing checkpoints.

Training neural baseline: zone_id_only_mlp | seed=42
[resume] zone_id_only_mlp seed=42 already completed. Reusing checkpoints.

Training neural baseline: zone_id_only_mlp | seed=123
[resume] zone_id_only_mlp seed=123 already completed. Reusing checkpoints.

Training neural bas

## 12. LightGBM CQR-style tabular baselines

The LightGBM baselines train three quantile regressors, one for each target quantile. Raw outputs are then passed through the same validation residual calibration utilities used by the graph-calibration notebooks. This gives a strong tabular probabilistic baseline without introducing a new graph architecture.


In [15]:
def lgbm_progress_callback(label: str) -> Callable:
    """Build a compact LightGBM single-line callback."""
    progress = ProgressPrinter(cfg.progress_min_interval_sec)

    def _callback(env):
        iteration = env.iteration + 1
        if iteration == 1 or iteration % 25 == 0:
            score_text = ""
            if env.evaluation_result_list:
                parts = []
                for data_name, eval_name, value, _higher_better in env.evaluation_result_list[:2]:
                    parts.append(f"{data_name}:{eval_name}={value:.5f}")
                score_text = " " + " ".join(parts)
            progress.update(f"[{label}] iter={iteration}{score_text}")

    _callback.order = 10
    return _callback


def train_lgbm_quantile_set(model_name: str, X: np.ndarray, feature_names: Sequence[str]) -> Optional[Dict[str, Any]]:
    """Train or resume three LightGBM quantile models."""
    if not LGB_AVAILABLE:
        skipped_baselines.append({"baseline": model_name, "reason": "LightGBM unavailable"})
        return None
    if X.shape[1] == 0:
        skipped_baselines.append({"baseline": model_name, "reason": "empty feature matrix"})
        return None

    model_dir = paths.models_dir / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    model_paths = {tau: model_dir / f"lgbm_tau_{str(tau).replace('.', '')}.pkl" for tau in [0.25, 0.50, 0.75]}
    meta_path = model_dir / "model_metadata.json"

    if cfg.resume and all(p.exists() for p in model_paths.values()) and meta_path.exists():
        models = {}
        for tau, path in model_paths.items():
            with path.open("rb") as file:
                models[tau] = pickle.load(file)
        print(f"[resume] Reusing LightGBM models for {model_name}")
        return {"models": models, "feature_names": list(feature_names), "model_name": model_name}

    X_train = X[mask_dict["train"]]
    X_val = X[mask_dict["val"]]
    y_train = y_np[mask_dict["train"]]
    y_val = y_np[mask_dict["val"]]
    w_train = sample_weight_np[mask_dict["train"]]
    w_val = sample_weight_np[mask_dict["val"]]

    models = {}
    for j, tau in enumerate([0.25, 0.50, 0.75]):
        label = f"{model_name} tau={tau}"
        reg = lgb.LGBMRegressor(
            objective="quantile",
            alpha=tau,
            n_estimators=cfg.lgbm_n_estimators,
            learning_rate=cfg.lgbm_learning_rate,
            num_leaves=cfg.lgbm_num_leaves,
            min_child_samples=cfg.lgbm_min_child_samples,
            subsample=cfg.lgbm_subsample,
            colsample_bytree=cfg.lgbm_colsample_bytree,
            random_state=cfg.split_seed + j,
            n_jobs=-1,
            verbose=-1,
        )
        callbacks = [
            lgb.early_stopping(cfg.lgbm_early_stopping_rounds, verbose=False),
            lgbm_progress_callback(label),
        ]
        reg.fit(
            X_train,
            y_train[:, j],
            sample_weight=w_train,
            eval_set=[(X_val, y_val[:, j])],
            eval_sample_weight=[w_val],
            eval_metric="quantile",
            callbacks=callbacks,
        )
        print(f"\n[{label}] best_iteration={getattr(reg, 'best_iteration_', None)}")
        models[tau] = reg
        atomic_pickle_dump(reg, model_paths[tau])

    atomic_write_json({
        "model_name": model_name,
        "feature_names": list(feature_names),
        "model_paths": {str(k): str(v) for k, v in model_paths.items()},
        "created_at": time.time(),
    }, meta_path)
    return {"models": models, "feature_names": list(feature_names), "model_name": model_name}


def predict_lgbm_quantile_set(bundle: Mapping[str, Any], X: np.ndarray, row_indices: np.ndarray) -> np.ndarray:
    """Predict q25/q50/q75 from a trained LightGBM quantile set."""
    models = bundle["models"]
    X_part = X[row_indices]
    pred = np.column_stack([
        models[0.25].predict(X_part),
        models[0.50].predict(X_part),
        models[0.75].predict(X_part),
    ]).astype(np.float32)
    return enforce_monotone_numpy(pred)


lgb_prediction_frames: List[pd.DataFrame] = []

lgb_specs = []
if "cqr_lgbm" in all_enabled:
    lgb_specs.append(("cqr_lgbm", X_base_topology if topology_feature_columns else X_base_prior, base_plus_topology_columns if topology_feature_columns else base_plus_prior_columns))
if cfg.run_appendix_baselines and "topology_only_lgbm" in all_enabled and topology_feature_columns:
    lgb_specs.append(("topology_only_lgbm", X_topology_prior, topology_feature_columns + COLD_PRIOR_COLUMNS))
if cfg.run_appendix_baselines and "demand_only_lgbm" in all_enabled and demand_feature_columns:
    lgb_specs.append(("demand_only_lgbm", X_demand_prior, demand_feature_columns + COLD_PRIOR_COLUMNS))

for model_name, X_matrix, feature_names in lgb_specs:
    print("\n" + "=" * 90)
    print(f"Training LightGBM baseline: {model_name}")
    print("=" * 90)
    bundle = train_lgbm_quantile_set(model_name, X_matrix, feature_names)
    if bundle is None:
        continue
    frames = []
    for split_name in ["val", "test"]:
        idx = split_indices_np[split_name]
        pred = predict_lgbm_quantile_set(bundle, X_matrix, idx)
        frames.append(make_prediction_frame(idx, pred, source="StrategicBaselines", model=model_name, checkpoint="raw", seed="lgbm"))
    raw_frame = pd.concat(frames, ignore_index=True)
    cal_frame = select_best_calibration(raw_frame, source="StrategicBaselines", model=model_name, seed="lgbm")
    lgb_prediction_frames.append(cal_frame)

lgb_predictions = pd.concat(lgb_prediction_frames, ignore_index=True) if lgb_prediction_frames else pd.DataFrame()
print("LightGBM predictions:", lgb_predictions.shape)



Training LightGBM baseline: cqr_lgbm

Training LightGBM baseline: topology_only_lgbm

Training LightGBM baseline: demand_only_lgbm
LightGBM predictions: (0, 0)


## 13. Prior baseline and seed ensembles

The Cold-prior prediction is the simplest non-graph anchor. Seed ensembles are constructed for neural baselines by averaging predictions across the five seeds for each `(model, checkpoint, split, row_id)` group.


In [16]:
# Cold-prior baseline predictions.
prior_frames = []
for split_name in ["val", "test"]:
    idx = split_indices_np[split_name]
    prior_frames.append(make_prediction_frame(idx, base_prior_np[idx], source="StrategicBaselines", model="cold_origin_dest_prior", checkpoint="raw", seed="prior"))
prior_predictions = pd.concat(prior_frames, ignore_index=True)

# Seed ensembles for neural baselines.
def build_seed_ensembles(frame: pd.DataFrame) -> pd.DataFrame:
    """Average neural predictions over seeds for each row/model/checkpoint/split."""
    if frame.empty:
        return pd.DataFrame()
    group_cols = ["source", "model", "checkpoint", "split", "row_id"]
    rows = []
    keep_cols = [
        "faf_orig_str", "faf_dest_str", "year_int", "true_q25", "true_q50", "true_q75", "sample_weight", "n_fmi_county_pairs",
    ]
    for keys, group in frame.groupby(group_cols, dropna=False):
        base = group.iloc[0]
        pred = group[["pred_q25", "pred_q50", "pred_q75"]].mean(axis=0).to_numpy(float)
        row = {col: base[col] for col in keep_cols if col in group.columns}
        for col, value in zip(group_cols, keys):
            row[col] = value
        row.update({"pred_q25": pred[0], "pred_q50": pred[1], "pred_q75": pred[2], "seed": "seed_ensemble"})
        rows.append(row)
    out = pd.DataFrame(rows)
    if not out.empty:
        out[["pred_q25", "pred_q50", "pred_q75"]] = enforce_monotone_numpy(out[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(float))
    return out

neural_ensembles = build_seed_ensembles(neural_predictions)

strategic_predictions = pd.concat(
    [frame for frame in [prior_predictions, neural_predictions, neural_ensembles, lgb_predictions] if not frame.empty],
    ignore_index=True,
) if any(not frame.empty for frame in [prior_predictions, neural_predictions, neural_ensembles, lgb_predictions]) else pd.DataFrame()

print("Prior predictions:", prior_predictions.shape)
print("Neural ensembles:", neural_ensembles.shape)
print("All strategic predictions:", strategic_predictions.shape)


Prior predictions: (2014, 17)
Neural ensembles: (18126, 17)
All strategic predictions: (110770, 17)


## 14. Load existing project predictions for direct comparison

This cell normalizes outputs from earlier notebooks into the same schema. It is intentionally permissive about column names because previous notebooks used slightly different naming conventions.


In [17]:
def normalize_external_prediction_frame(frame: pd.DataFrame, default_source: str) -> pd.DataFrame:
    """Normalize earlier notebook outputs to the common prediction schema."""
    clean = frame.copy()
    rename = {
        "truck_q25": "true_q25",
        "truck_q50": "true_q50",
        "truck_q75": "true_q75",
        "prediction_q25": "pred_q25",
        "prediction_q50": "pred_q50",
        "prediction_q75": "pred_q75",
        "q25_pred": "pred_q25",
        "q50_pred": "pred_q50",
        "q75_pred": "pred_q75",
        "predicted_q25": "pred_q25",
        "predicted_q50": "pred_q50",
        "predicted_q75": "pred_q75",
        "q25_hat": "pred_q25",
        "q50_hat": "pred_q50",
        "q75_hat": "pred_q75",
        "cold_split": "split",
        "cold_od_split": "split",
        "temporal_split": "split",
        "dataset_split": "split",
        "rowid": "row_id",
        "row_idx": "row_id",
        "edge_row_id": "row_id",
        "supervised_row_id": "row_id",
        "faf_orig": "faf_orig_str",
        "faf_dest": "faf_dest_str",
        "origin": "faf_orig_str",
        "destination": "faf_dest_str",
        "year": "year_int",
    }
    clean = clean.rename(columns={k: v for k, v in rename.items() if k in clean.columns})

    if "source" not in clean.columns:
        clean["source"] = default_source
    else:
        clean["source"] = default_source + ":" + clean["source"].astype(str)
    if "model" not in clean.columns:
        if "variant" in clean.columns:
            clean["model"] = clean["variant"].astype(str)
        else:
            clean["model"] = "unknown"
    if "checkpoint" not in clean.columns:
        if "checkpoint_metric" in clean.columns:
            clean["checkpoint"] = clean["checkpoint_metric"].astype(str)
        elif "checkpoint_name" in clean.columns:
            clean["checkpoint"] = clean["checkpoint_name"].astype(str)
        elif "variant" in clean.columns:
            clean["checkpoint"] = clean["variant"].astype(str)
        else:
            clean["checkpoint"] = "baseline"
    if "seed" not in clean.columns:
        clean["seed"] = "baseline"
    if "split" not in clean.columns:
        clean["split"] = "unknown"

    # Attach row_id and true labels when missing.
    if "row_id" not in clean.columns:
        clean = normalize_prediction_keys(clean)
        needed_keys = ["faf_orig_str", "faf_dest_str", "year_int"]
        if all(c in clean.columns for c in needed_keys):
            clean = clean.merge(
                supervised_df[["row_id", "faf_orig_str", "faf_dest_str", "year_int"] + LABEL_COLUMNS + (["n_fmi_county_pairs"] if "n_fmi_county_pairs" in supervised_df.columns else [])],
                on=needed_keys,
                how="left",
                suffixes=("", "_lookup"),
            )
    else:
        clean["row_id"] = pd.to_numeric(clean["row_id"], errors="coerce").astype("Int64")
        lookup_cols = ["row_id", "faf_orig_str", "faf_dest_str", "year_int"] + LABEL_COLUMNS
        if "n_fmi_county_pairs" in supervised_df.columns:
            lookup_cols.append("n_fmi_county_pairs")
        clean = clean.merge(supervised_df[lookup_cols], on="row_id", how="left", suffixes=("", "_lookup"))

    # Fill true labels and metadata from lookup columns if needed.
    for src_col, target_col in zip(LABEL_COLUMNS, ["true_q25", "true_q50", "true_q75"]):
        if target_col not in clean.columns:
            if src_col in clean.columns:
                clean[target_col] = clean[src_col]
            elif f"{src_col}_lookup" in clean.columns:
                clean[target_col] = clean[f"{src_col}_lookup"]
    for col in ["faf_orig_str", "faf_dest_str", "year_int", "n_fmi_county_pairs"]:
        lookup_col = f"{col}_lookup"
        if col not in clean.columns and lookup_col in clean.columns:
            clean[col] = clean[lookup_col]
        elif col in clean.columns and lookup_col in clean.columns:
            clean[col] = clean[col].where(clean[col].notna(), clean[lookup_col])

    required = ["source", "model", "checkpoint", "seed", "split", "row_id", "true_q25", "true_q50", "true_q75", "pred_q25", "pred_q50", "pred_q75"]
    missing = [c for c in required if c not in clean.columns]
    if missing:
        raise ValueError(f"External prediction frame missing required columns after normalization: {missing}")

    if "sample_weight" not in clean.columns:
        clean = clean.merge(pd.DataFrame({"row_id": supervised_df["row_id"], "sample_weight_lookup": sample_weight_np}), on="row_id", how="left")
        clean["sample_weight"] = clean["sample_weight_lookup"].fillna(1.0)
    if "n_fmi_county_pairs" not in clean.columns:
        clean["n_fmi_county_pairs"] = np.nan

    keep = ["source", "model", "checkpoint", "seed", "split", "row_id", "faf_orig_str", "faf_dest_str", "year_int", "true_q25", "true_q50", "true_q75", "pred_q25", "pred_q50", "pred_q75", "sample_weight", "n_fmi_county_pairs"]
    out = clean[keep].copy()
    out = out.dropna(subset=["row_id", "pred_q25", "pred_q50", "pred_q75"])
    out["row_id"] = out["row_id"].astype(int)
    out["split"] = out["split"].astype(str).str.lower()
    out = out[out["split"].isin(["val", "test"])]
    out[["pred_q25", "pred_q50", "pred_q75"]] = enforce_monotone_numpy(out[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(float))
    out = out.drop_duplicates(["source", "model", "checkpoint", "seed", "split", "row_id"], keep="first")
    return out.reset_index(drop=True)


external_frames = []
for name, path in paths.external_prediction_paths.items():
    if path.exists():
        try:
            raw = pd.read_parquet(path)
            norm = normalize_external_prediction_frame(raw, name)
            external_frames.append(norm)
            print(f"Loaded external predictions {name}: raw={raw.shape}, normalized={norm.shape}")
        except Exception as exc:
            skipped_baselines.append({"baseline": f"external:{name}", "reason": str(exc)})
            print(f"Warning: failed to normalize external predictions {name} from {path}: {exc}")
    else:
        print(f"[optional] External predictions not found for {name}: {path}")

external_predictions = pd.concat(external_frames, ignore_index=True) if external_frames else pd.DataFrame()
combined_predictions = pd.concat([frame for frame in [strategic_predictions, external_predictions] if not frame.empty], ignore_index=True) if (not strategic_predictions.empty or not external_predictions.empty) else pd.DataFrame()
print("External predictions:", external_predictions.shape)
print("Combined predictions:", combined_predictions.shape)


Loaded external predictions ColdOD_NonGraph: raw=(22154, 28), normalized=(22154, 17)
Loaded external predictions GraphV2_SAGE_HGT: raw=(94658, 31), normalized=(94658, 17)
Loaded external predictions DCQHGT_TopologyV3: raw=(515584, 20), normalized=(515584, 17)
Loaded external predictions DCQHGT_DisruptionGate: raw=(19283, 40), normalized=(7199, 17)
External predictions: (639595, 17)
Combined predictions: (750365, 17)


## 15. Decision-aware CVaR-style top-risk screening baselines

This table is separate from prediction metrics because some operational baselines are rankers rather than full quantile predictors. The same function also evaluates all full prediction models by using their predicted q75 and q75-q50 spread as the risk score.


In [18]:
def cvar_risk_score_from_quantiles(q25: np.ndarray, q50: np.ndarray, q75: np.ndarray, eta: float) -> np.ndarray:
    """Compute the paper's CVaR-style risk score from quantiles."""
    return q75 + eta * np.maximum(q75 - q50, 0.0)


def cvar_screening_from_scores(score: np.ndarray, true_q50: np.ndarray, true_q75: np.ndarray, top_fraction: float, eta: float) -> Dict[str, float]:
    """Evaluate top-risk screening regret and oracle overlap for one score vector."""
    score = np.asarray(score, dtype=float)
    true_risk = cvar_risk_score_from_quantiles(np.zeros_like(true_q50), true_q50, true_q75, eta)
    n = len(score)
    k = max(1, int(math.ceil(n * float(top_fraction))))
    pred_idx = np.argsort(-score)[:k]
    oracle_idx = np.argsort(-true_risk)[:k]
    pred_value = float(true_risk[pred_idx].sum())
    oracle_value = float(true_risk[oracle_idx].sum())
    regret = oracle_value - pred_value
    normalized_regret = regret / (oracle_value + 1.0e-12)
    overlap = len(set(pred_idx.tolist()) & set(oracle_idx.tolist())) / float(k)
    return {
        "top_fraction": float(top_fraction),
        "k": int(k),
        "oracle_value": oracle_value,
        "selected_value": pred_value,
        "regret": float(regret),
        "normalized_regret": float(normalized_regret),
        "oracle_overlap": float(overlap),
    }


def cvar_screening_for_predictions(predictions: pd.DataFrame, eta: float) -> pd.DataFrame:
    """Compute CVaR screening metrics for all full prediction models."""
    rows = []
    if predictions.empty:
        return pd.DataFrame()
    test = predictions.loc[predictions["split"].eq("test")].copy()
    group_cols = ["source", "model", "checkpoint", "seed"]
    for keys, group in test.groupby(group_cols, dropna=False):
        source, model, checkpoint, seed = keys
        score = cvar_risk_score_from_quantiles(
            group["pred_q25"].to_numpy(float),
            group["pred_q50"].to_numpy(float),
            group["pred_q75"].to_numpy(float),
            eta,
        )
        true_q50 = group["true_q50"].to_numpy(float)
        true_q75 = group["true_q75"].to_numpy(float)
        for frac in cfg.cvar_top_fractions:
            row = cvar_screening_from_scores(score, true_q50, true_q75, frac, eta)
            row.update({"source": source, "model": model, "checkpoint": checkpoint, "seed": seed, "score_type": "predicted_quantile_risk"})
            rows.append(row)
    return pd.DataFrame(rows)


def build_demand_volume_score(frame: pd.DataFrame) -> Optional[np.ndarray]:
    """Build a simple operational demand-volume rank score from demand-like columns."""
    if not demand_feature_columns:
        return None
    values = frame[demand_feature_columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
    med = np.nanmedian(values, axis=0)
    med = np.where(np.isfinite(med), med, 0.0)
    values = np.where(np.isfinite(values), values, med)
    std = np.nanstd(values, axis=0)
    std = np.where(std > 1.0e-6, std, 1.0)
    z = (values - np.nanmean(values, axis=0)) / std
    score = np.nanmean(np.maximum(z, 0.0), axis=1)
    return score.astype(float)


cvar_prediction_table = cvar_screening_for_predictions(combined_predictions, eta=cfg.cvar_eta)

# Rank-only operational baselines.
rank_rows = []
test_idx = split_indices_np["test"]
test_df = supervised_df.iloc[test_idx].copy()
true_q50_test = test_df["truck_q50"].to_numpy(float)
true_q75_test = test_df["truck_q75"].to_numpy(float)

rank_scores = {
    "cold_prior_q75_rank": base_prior_np[test_idx, 2].astype(float),
}
demand_score = build_demand_volume_score(test_df)
if demand_score is not None:
    rank_scores["demand_volume_rank"] = demand_score

for model_name, score in rank_scores.items():
    for frac in cfg.cvar_top_fractions:
        row = cvar_screening_from_scores(score, true_q50_test, true_q75_test, frac, cfg.cvar_eta)
        row.update({"source": "StrategicRankBaselines", "model": model_name, "checkpoint": "score_only", "seed": "baseline", "score_type": model_name})
        rank_rows.append(row)

# Random top-k baseline averaged over repeats.
rng = np.random.default_rng(cfg.split_seed)
for frac in cfg.cvar_top_fractions:
    repeated = []
    for r in range(cfg.random_rank_repeats):
        score = rng.normal(size=len(test_idx))
        repeated.append(cvar_screening_from_scores(score, true_q50_test, true_q75_test, frac, cfg.cvar_eta))
    repeated_df = pd.DataFrame(repeated)
    row = repeated_df.mean(numeric_only=True).to_dict()
    row.update({"source": "StrategicRankBaselines", "model": "random_rank", "checkpoint": f"mean_{cfg.random_rank_repeats}_runs", "seed": "random", "score_type": "random"})
    rank_rows.append(row)

rank_cvar_table = pd.DataFrame(rank_rows)
cvar_table = pd.concat([frame for frame in [cvar_prediction_table, rank_cvar_table] if not frame.empty], ignore_index=True)
if not cvar_table.empty:
    cvar_table = cvar_table.sort_values(["top_fraction", "normalized_regret", "oracle_overlap"], ascending=[True, True, False]).reset_index(drop=True)

print("CVaR prediction rows:", cvar_prediction_table.shape)
print("CVaR rank-only rows:", rank_cvar_table.shape)
cvar_table.head(20)


CVaR prediction rows: (1050, 12)
CVaR rank-only rows: (9, 12)


,top_fraction,k,oracle_value,selected_value,regret,normalized_regret,oracle_overlap,source,model,checkpoint,seed,score_type
0,0.05,53.0,4.574440e+05,2.460666e+05,211377.378950,0.462084,0.051132,StrategicRankBaselines,random_rank,mean_100_runs,random,random
1,0.05,53.0,4.574440e+05,4.345555e+05,22888.523926,0.050036,0.622642,DCQHGT_TopologyV3:DCQHGT_TOPOLOGY_V3,faf_hgt_no_topology,best_val_iqr,42,predicted_quantile_risk
2,0.05,53.0,4.574440e+05,4.332639e+05,24180.079468,0.052859,0.641509,DCQHGT_TopologyV3:DCQHGT_TOPOLOGY_V3,hgt_truck_plus_rail_adj,best_val_iqr,123,predicted_quantile_risk
3,0.05,53.0,4.574440e+05,4.332639e+05,24180.079468,0.052859,0.641509,DCQHGT_TopologyV3:DCQHGT_TOPOLOGY_V3,hgt_truck_plus_rail_adj,best_val_q75,123,predicted_quantile_risk
4,0.05,53.0,4.574440e+05,4.331512e+05,24292.773438,0.053105,0.584906,DCQHGT_TopologyV3:DCQHGT_TOPOLOGY_V3,hgt_terminal_plus_truck_plus_rail,best_val_q75,7,predicted_quantile_risk
5,0.05,53.0,4.574440e+05,4.331174e+05,24326.579346,0.053179,0.641509,DCQHGT_TopologyV3:DCQHGT_TOPOLOGY_V3,hgt_truck_plus_rail_adj,best_val_pinball,123,predicted_quantile_risk
6,0.05,53.0,4.574440e+05,4.326403e+05,24803.694092,0.054222,0.622642,DCQHGT_TopologyV3:DCQHGT_TOPOLOGY_V3,faf_hgt_no_topology,best_val_iqr,7,predicted_quantile_risk
7,0.05,53.0,4.574440e+05,4.317485e+05,25695.484497,0.056172,0.641509,DCQHGT_TopologyV3:DCQHGT_TOPOLOGY_V3_ENSEMBLE,faf_hgt_no_topology,best_val_iqr,ensemble,predicted_quantile_risk
8,0.05,53.0,4.574440e+05,4.316103e+05,25833.694824,0.056474,0.622642,DCQHGT_TopologyV3:DCQHGT_TOPOLOGY_V3,hgt_truck_plus_rail_adj,best_val_iqr,42,predicted_quantile_risk
9,0.05,53.0,4.574440e+05,4.316103e+05,25833.694824,0.056474,0.622642,DCQHGT_TopologyV3:DCQHGT_TOPOLOGY_V3,hgt_truck_plus_rail_adj,best_val_q75,42,predicted_quantile_risk


## 16. Metrics, paired summaries, and artifact writing

This cell writes all comparable results. The `combined_predictions` file includes both new strategic baselines and any previous project predictions found on disk.


In [19]:
strategic_metrics = compute_metrics_frame(strategic_predictions)
combined_metrics = compute_metrics_frame(combined_predictions)
strategic_leaderboard = build_leaderboard(strategic_metrics, split="test")
combined_leaderboard = build_leaderboard(combined_metrics, split="test")

# Paired summary against selected reference rows on strict test row_ids.
def paired_comparison(predictions: pd.DataFrame, reference_selectors: Sequence[Dict[str, str]]) -> pd.DataFrame:
    """Build paired row-level mean differences versus selected references."""
    if predictions.empty:
        return pd.DataFrame()
    test = predictions.loc[predictions["split"].eq("test")].copy()
    rows = []
    candidate_groups = ["source", "model", "checkpoint", "seed"]
    for selector in reference_selectors:
        ref = test.copy()
        for col, value in selector.items():
            if col in ref.columns:
                ref = ref.loc[ref[col].astype(str).str.contains(str(value), regex=False)]
        if ref.empty:
            continue
        # Use the best pinball row if the selector matches multiple model variants.
        ref_metrics = compute_metrics_frame(ref)
        ref_metrics = ref_metrics.loc[ref_metrics["split"].eq("test")].sort_values("pinball_mean")
        if ref_metrics.empty:
            continue
        ref_key = ref_metrics.iloc[0][candidate_groups].to_dict()
        ref_rows = test.copy()
        for col, value in ref_key.items():
            ref_rows = ref_rows.loc[ref_rows[col].astype(str).eq(str(value))]
        ref_rows = ref_rows[["row_id", "true_q25", "true_q50", "true_q75", "pred_q25", "pred_q50", "pred_q75"]].rename(
            columns={"pred_q25": "ref_pred_q25", "pred_q50": "ref_pred_q50", "pred_q75": "ref_pred_q75"}
        )
        ref_name = "|".join([f"{k}={v}" for k, v in ref_key.items()])
        for keys, group in test.groupby(candidate_groups, dropna=False):
            source, model, checkpoint, seed = keys
            if all(str(ref_key[col]) == str(val) for col, val in zip(candidate_groups, keys)):
                continue
            merged = group.merge(ref_rows, on=["row_id", "true_q25", "true_q50", "true_q75"], how="inner")
            if merged.empty:
                continue
            y_true = merged[["true_q25", "true_q50", "true_q75"]].to_numpy(float)
            pred = merged[["pred_q25", "pred_q50", "pred_q75"]].to_numpy(float)
            ref_pred = merged[["ref_pred_q25", "ref_pred_q50", "ref_pred_q75"]].to_numpy(float)
            pin = pinball_matrix(y_true, pred).mean(axis=1)
            ref_pin = pinball_matrix(y_true, ref_pred).mean(axis=1)
            rows.append({
                "reference": ref_name,
                "source": source,
                "model": model,
                "checkpoint": checkpoint,
                "seed": seed,
                "n_paired_rows": int(len(merged)),
                "mean_pinball_delta_vs_ref": float(np.mean(pin - ref_pin)),
                "median_pinball_delta_vs_ref": float(np.median(pin - ref_pin)),
                "win_rate_vs_ref": float(np.mean(pin < ref_pin)),
                "mean_q75_abs_delta_vs_ref": float(np.mean(np.abs(pred[:, 2] - y_true[:, 2]) - np.abs(ref_pred[:, 2] - y_true[:, 2]))),
                "mean_iqr_abs_delta_vs_ref": float(np.mean(np.abs((pred[:, 2] - pred[:, 0]) - (y_true[:, 2] - y_true[:, 0])) - np.abs((ref_pred[:, 2] - ref_pred[:, 0]) - (y_true[:, 2] - y_true[:, 0])))),
            })
    return pd.DataFrame(rows)

reference_selectors = [
    {"model": "ColdOD-MLP"},
    {"model": "HGT"},
    {"model": "GraphSAGE"},
    {"model": "cold_origin_dest_prior"},
]
paired_summary = paired_comparison(combined_predictions, reference_selectors)

# Output paths.
out_paths = {
    "strategic_predictions": paths.output_dir / "predictions_strategic_baselines_val_test.parquet",
    "combined_predictions": paths.output_dir / "combined_predictions_strategic_plus_existing_val_test.parquet",
    "strategic_metrics": paths.output_dir / "metrics_strategic_baselines.csv",
    "combined_metrics": paths.output_dir / "metrics_strategic_plus_existing.csv",
    "strategic_leaderboard": paths.output_dir / "leaderboard_test_strategic_baselines.csv",
    "combined_leaderboard": paths.output_dir / "leaderboard_test_strategic_plus_existing.csv",
    "cvar_table": paths.output_dir / "cvar_toprisk_screening_strategic_plus_existing.csv",
    "paired_summary": paths.output_dir / "paired_summary_strategic_plus_existing_test_only.csv",
    "neural_history": paths.output_dir / "training_history_neural_strategic_baselines.csv",
    "split_stats": paths.tables_dir / "strategic_baselines_cold_od_split_stats.csv",
    "skipped": paths.output_dir / "skipped_baselines.csv",
    "preprocessors": paths.output_dir / "preprocessors_strategic_baselines.json",
    "run_config": paths.output_dir / "run_config_strategic_baselines.json",
    "artifact_manifest": paths.output_dir / "analysis_artifact_manifest_strategic_baselines.json",
}

write_dataframe(strategic_predictions, out_paths["strategic_predictions"])
write_dataframe(combined_predictions, out_paths["combined_predictions"])
write_dataframe(strategic_metrics, out_paths["strategic_metrics"])
write_dataframe(combined_metrics, out_paths["combined_metrics"])
write_dataframe(strategic_leaderboard, out_paths["strategic_leaderboard"])
write_dataframe(combined_leaderboard, out_paths["combined_leaderboard"])
write_dataframe(cvar_table, out_paths["cvar_table"])
write_dataframe(paired_summary, out_paths["paired_summary"])
write_dataframe(neural_history, out_paths["neural_history"])
write_dataframe(split_stats_df, out_paths["split_stats"])
write_dataframe(pd.DataFrame(skipped_baselines), out_paths["skipped"])

atomic_write_json({name: prep.to_dict() for name, prep in preprocessors.items()}, out_paths["preprocessors"])

run_config_payload = {
    "notebook": "Strategic_Baselines_ColdOD_Comparison.ipynb",
    "created_at_unix": time.time(),
    "config": asdict(cfg),
    "paths": {k: str(v) if not isinstance(v, Mapping) else {kk: str(vv) for kk, vv in v.items()} for k, v in asdict(paths).items()},
    "split_source": split_source,
    "dataset": {
        "n_rows": int(len(supervised_df)),
        "split_counts": supervised_df["cold_split"].value_counts(dropna=False).to_dict(),
        "manifest_feature_count": int(len(manifest_feature_columns)),
        "topology_feature_count": int(len(topology_feature_columns)),
        "demand_feature_count": int(len(demand_feature_columns)),
        "target_scale": float(target_scale),
    },
    "package_versions": {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "torch": None if not TORCH_AVAILABLE else torch.__version__,
        "pyg_available": PYG_AVAILABLE,
        "lightgbm_available": LGB_AVAILABLE,
    },
}
atomic_write_json(run_config_payload, out_paths["run_config"])
atomic_write_json({k: str(v) for k, v in out_paths.items()}, out_paths["artifact_manifest"])

print("Saved strategic baseline artifacts to:", paths.output_dir)
print(json.dumps({k: str(v) for k, v in out_paths.items()}, indent=2))


Saved strategic baseline artifacts to: E:\NetworkOptimization\pythonProject1\Data\10_experiments\strategic_cold_od_baselines_v1_notebook\east_plus_gulf
{
  "strategic_predictions": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\strategic_cold_od_baselines_v1_notebook\\east_plus_gulf\\predictions_strategic_baselines_val_test.parquet",
  "combined_predictions": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\strategic_cold_od_baselines_v1_notebook\\east_plus_gulf\\combined_predictions_strategic_plus_existing_val_test.parquet",
  "strategic_metrics": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\strategic_cold_od_baselines_v1_notebook\\east_plus_gulf\\metrics_strategic_baselines.csv",
  "combined_metrics": "E:\\NetworkOptimization\\pythonProject1\\Data\\10_experiments\\strategic_cold_od_baselines_v1_notebook\\east_plus_gulf\\metrics_strategic_plus_existing.csv",
  "strategic_leaderboard": "E:\\NetworkOptimization\\pythonProject1\\Data\\10

## 17. Compact report preview

The report below is also written as a Markdown file under `reports/`. Use it when updating the paper tables.


In [22]:
report_lines = []
report_lines.append("# Strategic Cold-OD Baseline Report")
report_lines.append("")
report_lines.append(f"Output directory: `{paths.output_dir}`")
report_lines.append(f"Split source: `{split_source}`")
report_lines.append("")
report_lines.append("## Cold-OD split statistics")
report_lines.append("")
report_lines.append(split_stats_df.to_markdown(index=False))
report_lines.append("")
report_lines.append("## Strategic baseline leaderboard, test split")
report_lines.append("")
if strategic_leaderboard.empty:
    report_lines.append("No strategic leaderboard rows were produced.")
else:
    cols = ["source", "model", "checkpoint", "seed", "n_rows", "pinball_mean", "weighted_pinball_mean", "mae_q50", "mae_q75", "iqr_mae", "stress_top10_mae_q75", "sparse_bottom25_mae_q75"]
    report_lines.append(strategic_leaderboard[[c for c in cols if c in strategic_leaderboard.columns]].head(30).to_markdown(index=False))
report_lines.append("")
report_lines.append("## Combined leaderboard with prior project predictions, test split")
report_lines.append("")
if combined_leaderboard.empty:
    report_lines.append("No combined leaderboard rows were produced.")
else:
    cols = ["source", "model", "checkpoint", "seed", "n_rows", "pinball_mean", "weighted_pinball_mean", "mae_q50", "mae_q75", "iqr_mae", "stress_top10_mae_q75", "sparse_bottom25_mae_q75"]
    report_lines.append(combined_leaderboard[[c for c in cols if c in combined_leaderboard.columns]].head(40).to_markdown(index=False))
report_lines.append("")
report_lines.append("## CVaR-style top-risk screening")
report_lines.append("")
if cvar_table.empty:
    report_lines.append("No CVaR rows were produced.")
else:
    cols = ["top_fraction", "source", "model", "checkpoint", "seed", "score_type", "normalized_regret", "oracle_overlap", "regret"]
    report_lines.append(cvar_table[[c for c in cols if c in cvar_table.columns]].head(60).to_markdown(index=False))
report_lines.append("")
report_lines.append("## Skipped baselines")
report_lines.append("")
if skipped_baselines:
    report_lines.append(pd.DataFrame(skipped_baselines).to_markdown(index=False))
else:
    report_lines.append("None.")

report_text = "\n".join(report_lines)
report_path = paths.reports_dir / "strategic_cold_od_baseline_report.md"
report_path.write_text(report_text, encoding="utf-8")
print(report_text[:5000])
print("\nReport written to:", report_path)


# Strategic Cold-OD Baseline Report

Output directory: `E:\NetworkOptimization\pythonProject1\Data\10_experiments\strategic_cold_od_baselines_v1_notebook\east_plus_gulf`
Split source: `external_predictions:E:\NetworkOptimization\pythonProject1\Data\10_experiments\cold_od_split_baselines_v1_notebook\east_plus_gulf\predictions_cold_od_all_splits.parquet`

## Cold-OD split statistics

| split   |   n_rows |   n_od_pairs | years                              |   n_origin_zones |   n_dest_zones |
|:--------|---------:|-------------:|:-----------------------------------|-----------------:|---------------:|
| train   |    42849 |         8748 | 2018,2019,2020,2021,2022           |              104 |            104 |
| val     |      957 |          957 | 2023                               |              104 |            104 |
| test    |     1057 |         1057 | 2024                               |              104 |            104 |
| unused  |    29109 |        10631 | 2018,2019,2020,2021,20